# 07 · LLM: prompt, risposte con fonti e valutazione

**Progetto:** HelpDesk AI · **Passo 3:** Generative AI

## Obiettivo di questo notebook

Nel notebook 06 abbiamo costruito la ricerca, cioè la "R" di RAG (*Retrieval*). Ora entra in gioco l'LLM, la "G" (*Generation*): capisce la richiesta, scrive le istruzioni a partire dai documenti trovati e prepara il ticket quando serve un tecnico.

```
richiesta in italiano
  -> analisi (LLM): traduzione, dispositivo, categoria, codice di errore
  -> dispositivo non chiaro? l'app lo chiede all'utente
  -> ricerca nell'indice filtrata per dispositivo (con HyDE, se migliora)
  -> risposta (LLM): istruzioni con fonti, oppure una domanda, oppure l'inoltro al tecnico
  -> problema non risolto? riepilogo del ticket e riclassificazione (LLM)
```

Ogni passaggio viene misurato. Dove serve leggere molti testi usiamo un secondo LLM come giudice ma prima verifichiamo che giudichi come noi.

## Cosa faremo

1. Import e configurazione
2. Richieste di prova, indice e classificatore
3. La funzione per chiamare l'LLM
4. Analisi della richiesta (prompt 1)
5. Ricerca con l'analisi dell'LLM
6. Il giudice dei chunk: prima lo verifichiamo
7. HyDE (prompt 2): cercare con una risposta ipotetica
8. La risposta del chatbot (prompt 3)
9. Il giudice delle risposte (prompt 4)
10. Riepilogo del ticket e riclassificazione (prompt 5)
11. Costi e tempi
12. Scelte per l'app e salvataggio
13. Riepilogo

## 1. Import e configurazione

| Libreria | A cosa serve |
|---|---|
| `openai` | chiamare l'LLM con la *Responses API* e creare gli embeddings |
| `pydantic` | descrivere la forma delle risposte che vogliamo dall'LLM (installata insieme a `openai`) |
| `chromadb` | riaprire l'indice del notebook 06 |
| `concurrent.futures` | fare più chiamate all'API contemporaneamente |
| `hashlib` | calcolare le chiavi della cache delle risposte |
| `sklearn` | misurare kappa di Cohen tra il giudice e i nostri giudizi |

### Perché due modelli

| Ruolo | Modello | Prezzo (dollari per milione di token: ingresso / uscita) | Perché |
|---|---|---|---|
| chatbot | `gpt-6-luna` | 0,10 / 0,50 | il più economico e veloce della generazione attuale, pensato per compiti mirati e ripetuti: nell'app ogni conversazione fa 3-4 chiamate |
| giudice | `gpt-6.1-terra` | 1,00 / 5,00 | il giudice deve essere **diverso** e **più capace** del modello che giudica: un modello tende a promuovere i testi scritti da lui stesso. Costa 10 volte tanto ma lo usiamo solo qui |

### Perché la Responses API

È l'interfaccia che OpenAI consiglia per i modelli più recenti e permette le **risposte strutturate**: invece di un testo libero, l'LLM restituisce dei campi con una forma decisa da noi.

### Lo sforzo di ragionamento

I modelli GPT-6 **ragionano** prima di rispondere: producono dei token nascosti che si pagano come token in uscita. Il parametro `effort` decide quanto: `none`, `low`, `medium` (il valore predefinito), `high`, ... Usiamo:
- `none` per i compiti semplici (analisi, HyDE, riepilogo): più veloce ed economico;
- `low` per la risposta all'utente e per il giudice dove bisogna applicare delle regole.

In [1]:
from pathlib import Path
import os
import json
import time
import hashlib
import re
import threading
from concurrent.futures import ThreadPoolExecutor
from typing import Literal, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from sklearn.metrics import cohen_kappa_score

In [2]:
# --- Percorsi ---
PROJECT_DIR = Path("..")
KB_DIR = PROJECT_DIR / "knowledge_base"
MODELS_DIR = PROJECT_DIR / "models"
RICHIESTE_PATH = KB_DIR / "richieste_prova.csv"
GIUDIZI_PATH = KB_DIR / "giudizi_manuali.csv"                     # i nostri giudizi della sezione 9.4 del notebook 06
EMBEDDINGS_RICHIESTE_PATH = KB_DIR / "embeddings_richieste.npz"   # dal notebook 06
CHROMA_DIR = KB_DIR / "chroma"
CONFIG_RAG_PATH = MODELS_DIR / "configurazione_rag.json"
CONFIG_CLASSIFICATORE_PATH = MODELS_DIR / "configurazione_modello.json"
CONFIG_LLM_PATH = MODELS_DIR / "configurazione_llm.json"
CACHE_LLM_PATH = KB_DIR / "cache_llm.jsonl"                        # risposte dell'LLM già ottenute (escluso da Git)
CACHE_EMBEDDINGS_PATH = KB_DIR / "cache_embeddings.npz"            # embeddings dei testi scritti dall'LLM (escluso da Git)

# --- Modelli e prezzi (dollari per milione di token, novembre 2026) ---
MODELLO_LLM = "gpt-6-luna"          # tutti i compiti del chatbot
MODELLO_GIUDICE = "gpt-6.1-terra"   # la valutazione: un modello diverso e più capace
PREZZI = {                          # (ingresso, ingresso già in cache, uscita)
    "gpt-6-luna": (0.10, 0.01, 0.50),
    "gpt-6.1-terra": (1.00, 0.05, 5.00),
}
MAX_THREAD = 8                      # chiamate all'API contemporanee

# --- Stile dei grafici ---
COLORE = "#2a78d6"
TESTO_SECONDARIO = "#52514e"
GRIGLIA = "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRIGLIA,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": GRIGLIA,
    "xtick.color": TESTO_SECONDARIO,
    "ytick.color": TESTO_SECONDARIO,
})
pd.set_option("display.max_colwidth", 120)

## 2. Richieste di prova, indice e classificatore

Ricarichiamo tutto quello che serve dai notebook precedenti:

- la **configurazione della ricerca** scelta nel notebook 06 (collezione, k, nomi dei dispositivi in inglese);
- l'**indice** ChromaDB;
- le **36 richieste di prova** con i nostri **giudizi** della sezione 9.4 (file `giudizi_manuali.csv`);
- gli **embeddings delle traduzioni manuali** salvati nel notebook 06: con questi ritroviamo esattamente gli stessi risultati;
- il **classificatore SVM** del notebook 04 per confrontarlo con l'LLM.

In [3]:
with open(CONFIG_RAG_PATH, encoding="utf-8") as f:
    configurazione_rag = json.load(f)
MODELLO_EMBEDDINGS = configurazione_rag["modello_embeddings"]
K = configurazione_rag["k"]
DISPOSITIVO_IN_INGLESE = configurazione_rag["dispositivo_in_inglese"]

collezione = chromadb.PersistentClient(path=str(CHROMA_DIR)).get_collection(configurazione_rag["collezione"])
print(f"Indice: collezione '{collezione.name}' con {collezione.count()} chunk | k = {K}")

richieste = pd.read_csv(RICHIESTE_PATH)
richieste["fonti_attese"] = richieste["fonti_attese"].str.split("|")
richieste = richieste.merge(pd.read_csv(GIUDIZI_PATH), on="id", how="left")
assert richieste["giudizio_manuale"].notna().all(), "Manca il giudizio manuale per qualche richiesta"
print(f"Richieste di prova: {len(richieste)}")
display(richieste["giudizio_manuale"].value_counts().to_frame("giudizi manuali (notebook 06, sezione 9.4)"))

with np.load(EMBEDDINGS_RICHIESTE_PATH) as salvati:
    assert salvati["testi_en"].tolist() == richieste["richiesta_en"].tolist(), "Le richieste sono cambiate: riesegui il notebook 06"
    vettori_en_manuale = salvati["en"]

Indice: collezione 'kb_intestazione' con 3065 chunk | k = 5
Richieste di prova: 36


,"giudizi manuali (notebook 06, sezione 9.4)"
giudizio_manuale,
sì,17
per il tecnico,11
parziale,6
no,2


In [4]:
# Il classificatore SVM del notebook 04
with open(CONFIG_CLASSIFICATORE_PATH, encoding="utf-8") as f:
    configurazione_classificatore = json.load(f)
classificatore = joblib.load(MODELS_DIR / configurazione_classificatore["file_modello"])
SOGLIA = configurazione_classificatore["soglia_sicurezza"]
CATEGORIE = list(configurazione_classificatore["classi"])


def prepara_per_classificatore(testo):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z\s]", " ", testo.lower())).strip()


probabilita = classificatore.predict_proba(richieste["richiesta_en"].map(prepara_per_classificatore))
richieste["categoria_svm"] = classificatore.classes_[probabilita.argmax(axis=1)]
richieste["sicurezza_svm"] = probabilita.max(axis=1).round(2)
print(f"SVM: categoria corretta {(richieste['categoria_svm'] == richieste['categoria_attesa']).sum()} su {len(richieste)}")

SVM: categoria corretta 19 su 36


In [5]:
def cerca(vettore, dispositivo=None, k=K):
    # I k chunk più vicini, eventualmente solo di un dispositivo: metadati, distanza e testo
    risultato = collezione.query(query_embeddings=[vettore.tolist()], n_results=k,
                                 where={"dispositivo": dispositivo} if dispositivo else None,
                                 include=["metadatas", "distances", "documents"])
    return [{**m, "distanza": d, "testo": t}
            for m, d, t in zip(risultato["metadatas"][0], risultato["distances"][0], risultato["documents"][0])]


def valuta(fonti_trovate, fonti_attese):
    # Le stesse misure del notebook 06, a livello di documento
    giuste = [fonte in fonti_attese for fonte in fonti_trovate]
    posizione = giuste.index(True) + 1 if any(giuste) else None
    return {
        "trovata": posizione is not None,
        "reciprocal_rank": 1 / posizione if posizione else 0.0,
        "precisione": sum(giuste) / len(giuste) if giuste else 0.0,
    }


def testo_brani(trovati):
    # I chunk trovati, numerati come li vedrà l'LLM: [1], [2], ...
    blocchi = []
    for n, t in enumerate(trovati, start=1):
        dove = f"pagina {t['pagina']}" if t["pagina"] else f"sezione: {t['sezione']}"
        blocchi.append(f"[{n}] {t['editore']} - {t['titolo']} ({dove})\n{t['testo']}")
    return "\n\n".join(blocchi)

## 3. La funzione per chiamare l'LLM

Tutte le chiamate passano da `chiedi()` che fa quattro cose:

**1. Risposte strutturate.** Le descriviamo con una classe Pydantic: per esempio un campo `dispositivo` che può valere solo uno degli 8 dispositivi o `non chiaro`. L'API garantisce che la risposta rispetti lo schema. All'app non serve un testo libero da interpretare ma dei campi da usare: il dispositivo per il filtro, l'esito per decidere cosa mostrare.

**2. Cache su file.** Ogni risposta viene salvata in `cache_llm.jsonl` con una chiave calcolata da modello, sforzo, istruzioni, testo e schema. Rieseguendo il notebook la stessa domanda non viene rifatta: nessuna spesa e risultati identici. L'LLM, come gli embeddings, non risponde sempre allo stesso modo, quindi, se cambiamo anche una sola parola del prompt, la chiave cambia e la risposta viene richiesta di nuovo.

**3. Token e tempi.** Per ogni chiamata salviamo token in ingresso, in uscita, di ragionamento e secondi: nella sezione 11 calcoleremo quanto costa e quanto dura una conversazione.

**4. `store=False`.** Chiediamo a OpenAI di non conservare le risposte sui suoi server dopo la chiamata.

Con `in_parallelo()` facciamo fino a 8 chiamate insieme: 36 richieste in poche decine di secondi invece che in alcuni minuti. Se superiamo i limiti dell'API, la libreria `openai` aspetta e riprova da sola (`max_retries`).

In [6]:
load_dotenv(PROJECT_DIR / ".env")
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Chiave non trovata: crea il file .env nella cartella del progetto con OPENAI_API_KEY=...")
client = OpenAI(max_retries=5)


def carica_cache_llm():
    cache = {}
    if CACHE_LLM_PATH.exists():
        with open(CACHE_LLM_PATH, encoding="utf-8") as f:
            for riga in f:
                record = json.loads(riga)
                cache[record["chiave"]] = record
    return cache


cache_llm = carica_cache_llm()
blocco_cache = threading.Lock()   # più chiamate in parallelo scrivono nello stesso file: una alla volta
chiavi_usate = set()              # le chiamate di questa esecuzione (per i costi della sezione 11)
print(f"Risposte dell'LLM già in cache: {len(cache_llm)}")


def chiedi(compito, istruzioni, testo, formato, modello=MODELLO_LLM, sforzo="none"):
    # Risposta strutturata secondo `formato` (una classe Pydantic). Se la domanda è già stata fatta,
    # restituisce la risposta salvata: nessuna spesa e stesso risultato.
    schema = json.dumps(formato.model_json_schema(), sort_keys=True)
    chiave = hashlib.sha256(json.dumps([modello, sforzo, istruzioni, testo, schema]).encode("utf-8")).hexdigest()
    with blocco_cache:
        chiavi_usate.add(chiave)
    if chiave in cache_llm:
        return formato.model_validate(cache_llm[chiave]["risultato"])

    inizio = time.perf_counter()
    risposta = client.responses.parse(
        model=modello, instructions=istruzioni, input=testo, text_format=formato,
        reasoning={"effort": sforzo}, max_output_tokens=8000, store=False,
    )
    if risposta.status != "completed" or risposta.output_parsed is None:
        raise RuntimeError(f"Risposta non valida per '{compito}': stato {risposta.status}, {risposta.incomplete_details}")

    uso = risposta.usage
    record = {
        "chiave": chiave, "compito": compito, "modello": modello, "sforzo": sforzo,
        "risultato": risposta.output_parsed.model_dump(),
        "token_ingresso": uso.input_tokens,
        "token_in_cache": uso.input_tokens_details.cached_tokens,
        "token_uscita": uso.output_tokens,
        "token_ragionamento": uso.output_tokens_details.reasoning_tokens,
        "secondi": round(time.perf_counter() - inizio, 2),
    }
    with blocco_cache:
        cache_llm[chiave] = record
        with open(CACHE_LLM_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return risposta.output_parsed


def in_parallelo(funzione, elementi):
    # funzione(elemento) per ogni elemento, con più chiamate all'API contemporaneamente; l'ordine resta quello di partenza
    with ThreadPoolExecutor(max_workers=MAX_THREAD) as esecutore:
        return list(esecutore.map(funzione, elementi))


def costo(record):
    # Costo stimato di una chiamata in dollari (il costo esatto è nella dashboard di OpenAI)
    prezzo_ingresso, prezzo_cache, prezzo_uscita = PREZZI[record["modello"]]
    non_in_cache = record["token_ingresso"] - record["token_in_cache"]
    return (non_in_cache * prezzo_ingresso + record["token_in_cache"] * prezzo_cache
            + record["token_uscita"] * prezzo_uscita) / 1e6

Risposte dell'LLM già in cache: 503


In [7]:
# Embeddings con cache: lo stesso testo dà sempre lo stesso vettore (e non si paga due volte)
def carica_cache_embeddings():
    if CACHE_EMBEDDINGS_PATH.exists():
        with np.load(CACHE_EMBEDDINGS_PATH) as salvati:
            return {chiave: salvati[chiave] for chiave in salvati.files}
    return {}


cache_embeddings = carica_cache_embeddings()


def chiave_embedding(testo):
    return hashlib.sha256((MODELLO_EMBEDDINGS + testo).encode("utf-8")).hexdigest()


def embeddings(testi):
    mancanti = list(dict.fromkeys(t for t in testi if chiave_embedding(t) not in cache_embeddings))
    if mancanti:
        risposta = client.embeddings.create(model=MODELLO_EMBEDDINGS, input=mancanti)
        for testo, dato in zip(mancanti, risposta.data):
            cache_embeddings[chiave_embedding(testo)] = np.array(dato.embedding, dtype=np.float32)
        np.savez(CACHE_EMBEDDINGS_PATH, **cache_embeddings)
    return np.array([cache_embeddings[chiave_embedding(t)] for t in testi])


print(f"Embeddings già in cache: {len(cache_embeddings)}")

Embeddings già in cache: 108


## 4. Analisi della richiesta (prompt 1)

Con una sola chiamata chiediamo quattro cose:

| Campo | A cosa serve nell'app |
|---|---|
| `richiesta_en` | la traduzione in inglese per la ricerca (nel notebook 06 fatta a mano) |
| `dispositivo` | il filtro della ricerca; se è `non chiaro`, l'app mostra i pulsanti per sceglierlo |
| `categoria` | il gruppo a cui inoltrare il ticket: lo confrontiamo con l'SVM |
| `codice_errore` | il codice citato dall'utente, utile per la risposta e per il ticket |

**Perché una chiamata sola?** Quattro chiamate separate vorrebbero dire quattro attese e quattro costi.

**Le scelte nel prompt:**
- **"non chiaro" invece di indovinare:** un dispositivo sbagliato filtra i manuali sbagliati e nasconde la risposta giusta mentre una domanda costa all'utente un clic;
- **le descrizioni dei dispositivi e delle categorie** dicono all'LLM cosa significa ogni etichetta. Quelle delle categorie riprendono il significato che hanno nel dataset dei ticket. Le abbiamo scritte prima di vedere i risultati e non le ritocchiamo guardando le 36 richieste: altrimenti adatteremmo il prompt al set di validazione;
- **un piccolo glossario** ("libretto" -> "passbook") per la traduzione: è il vocabolario tipico dell'ambiente lavorativo, come quello che un'azienda darebbe a un nuovo collega.

In [8]:
DESCRIZIONE_DISPOSITIVI = {
    "computer": "il PC Windows dell'utente: programmi, account e accesso, rete e Wi-Fi, VPN, cartelle condivise",
    "stampante di sportello": "la stampante collegata via cavo al PC per stampa e scansione di documenti e/o libretti (Olivetti MB2)",
    "stampante di rete": "la stampante multifunzione di rete dell'ufficio (Kyocera, Epson, Lexmark): stampa, copia, scansione",
    "stampante di etichette": "la stampante di etichette (Zebra)",
    "POS": "il terminale di pagamento con carta",
    "pad di firma": "la tavoletta su cui il cliente firma (Wacom, Namirial)",
    "monitor": "lo schermo collegato al computer",
    "selezionatrice di banconote": "la contabanconote / selezionatrice di banconote (JetScan)",
}
DESCRIZIONE_CATEGORIE = {
    "Hardware": "guasti e malfunzionamenti di dispositivi fisici e periferiche (computer, stampanti, monitor, terminali, lettori) e del loro collegamento",
    "Access": "accesso a sistemi e applicazioni: account, password, PIN, autenticazione a più fattori, account bloccati o disabilitati",
    "Administrative rights": "operazioni che richiedono diritti di amministratore sul computer: installare o aggiornare programmi, aggiornamenti di sistema, impostazioni di sistema",
    "Storage": "file e cartelle: cartelle condivise, unità di rete, spazio su disco, permessi sui file, backup",
    "HR Support": "questioni del personale: buste paga, ferie, contratti, assunzioni e cessazioni",
    "Purchase": "acquisto di nuove apparecchiature, programmi o licenze, ordini e preventivi",
    "Internal Project": "attività legate a progetti interni dell'azienda",
    "Miscellaneous": "tutto ciò che non rientra nelle altre categorie",
}
assert set(DESCRIZIONE_DISPOSITIVI) == set(DISPOSITIVO_IN_INGLESE), "I dispositivi devono essere quelli dell'indice"
assert set(DESCRIZIONE_CATEGORIE) == set(CATEGORIE), "Le categorie devono essere quelle del classificatore"
DISPOSITIVI = list(DESCRIZIONE_DISPOSITIVI)

def elenco(descrizioni):
    return "\n".join(f"   - {nome}: {testo}" for nome, testo in descrizioni.items())

class AnalisiRichiesta(BaseModel):
    richiesta_en: str = Field(description="Traduzione fedele della richiesta in inglese")
    dispositivo: Literal[tuple(DISPOSITIVI + ["non chiaro"])]
    categoria: Literal[tuple(CATEGORIE)]
    codice_errore: Optional[str] = Field(description="Il codice di errore citato dall'utente, esattamente come scritto; null se non c'è")

ISTRUZIONI_ANALISI = f'''Sei il primo livello dell'help desk IT di un'azienda con uffici aperti al pubblico.
Ricevi la richiesta di un impiegato, scritta in italiano, e la analizzi per il sistema di ticketing.

1. Traduci la richiesta in inglese in modo fedele, senza aggiungere né togliere informazioni. Usa il lessico dei manuali tecnici in inglese (per esempio "libretto" -> "passbook", "contabanconote" -> "banknote counter"). Lascia invariati codici di errore e nomi di prodotti.
2. Indica il dispositivo di cui parla la richiesta, scegliendo tra:
{elenco(DESCRIZIONE_DISPOSITIVI)}
   Rispondi "non chiaro" se la richiesta può riferirsi a più dispositivi dell'elenco (per esempio "la stampante" senza dire quale). Meglio chiedere che indovinare: l'app chiederà all'utente.
3. Indica la categoria del ticket, scegliendo tra:
{elenco(DESCRIZIONE_CATEGORIE)}
4. Riporta il codice di errore se l'utente ne cita uno, altrimenti null.'''

print(ISTRUZIONI_ANALISI)

Sei il primo livello dell'help desk IT di un'azienda con uffici aperti al pubblico.
Ricevi la richiesta di un impiegato, scritta in italiano, e la analizzi per il sistema di ticketing.

1. Traduci la richiesta in inglese in modo fedele, senza aggiungere né togliere informazioni. Usa il lessico dei manuali tecnici in inglese (per esempio "libretto" -> "passbook", "contabanconote" -> "banknote counter"). Lascia invariati codici di errore e nomi di prodotti.
2. Indica il dispositivo di cui parla la richiesta, scegliendo tra:
   - computer: il PC Windows dell'utente: programmi, account e accesso, rete e Wi-Fi, VPN, cartelle condivise
   - stampante di sportello: la stampante collegata via cavo al PC per stampa e scansione di documenti e/o libretti (Olivetti MB2)
   - stampante di rete: la stampante multifunzione di rete dell'ufficio (Kyocera, Epson, Lexmark): stampa, copia, scansione
   - stampante di etichette: la stampante di etichette (Zebra)
   - POS: il terminale di pagamento con cart

In [9]:
%%time
def analizza(r):
    return chiedi("analisi", ISTRUZIONI_ANALISI, r["richiesta_it"], AnalisiRichiesta)

analisi = in_parallelo(analizza, richieste.to_dict("records"))
richieste["richiesta_en_llm"] = [a.richiesta_en for a in analisi]
richieste["dispositivo_llm"] = [a.dispositivo for a in analisi]
richieste["categoria_llm"] = [a.categoria for a in analisi]
richieste["codice_errore_llm"] = [a.codice_errore for a in analisi]
display(richieste[["id", "richiesta_it", "richiesta_en", "richiesta_en_llm"]].head(8))

,id,richiesta_it,richiesta_en,richiesta_en_llm
0,R01,La stampante non acquisisce automaticamente i dati dai fogli scansionati,The printer doesn't automatically capture the data from the scanned sheets,The printer does not automatically capture data from the scanned sheets.
1,R02,Si è incastrato un foglio dentro la stampante di sportello,A sheet got stuck inside the counter printer,A sheet of paper is jammed inside the counter printer.
2,R03,La stampa sui libretti è sbiadita e non si legge quasi più,The printing on the passbooks is faded and barely readable anymore,Printing on the passbooks is faded and is almost unreadable.
3,R04,"Il POS non è collegato, dà errore di connessione","The POS is not connected, it gives a connection error",The POS is not connected; it gives a connection error.
4,R05,"Il POS non stampa lo scontrino, esce il foglio bianco","The POS doesn't print the receipt, the paper comes out blank",The POS does not print the receipt; a blank sheet comes out.
5,R06,Il cliente inserisce la carta ma il POS non la legge,The customer inserts the card but the POS doesn't read it,"The customer inserts the card, but the POS does not read it."
6,R07,Il pad per la firma ha lo schermo spento,The signature pad screen is off,The signature pad screen is off.
7,R08,Il cliente firma sul pad ma la firma non compare,The customer signs on the pad but the signature doesn't appear,"The customer signs on the pad, but the signature does not appear."


CPU times: total: 15.6 ms
Wall time: 17 ms


### 4.1 Il dispositivo

Tre esiti possibili:
- **giusto:** il filtro userà i manuali giusti;
- **non chiaro:** l'app chiede all'utente (un clic in più ma nessun errore);
- **sbagliato:** il caso peggiore perché il filtro nasconde i manuali giusti.

In [10]:
def esito_dispositivo(r):
    if r["dispositivo_llm"] == r["dispositivo_atteso"]:
        return "giusto"
    if r["dispositivo_llm"] == "non chiaro":
        return "non chiaro (l'app chiede)"
    return "sbagliato"


richieste["esito_dispositivo"] = richieste.apply(esito_dispositivo, axis=1)
display(richieste["esito_dispositivo"].value_counts().to_frame("richieste"))
display(richieste.loc[richieste["esito_dispositivo"] != "giusto",
                      ["id", "richiesta_it", "dispositivo_atteso", "dispositivo_llm"]])

,richieste
esito_dispositivo,
giusto,34
non chiaro (l'app chiede),2


,id,richiesta_it,dispositivo_atteso,dispositivo_llm
0,R01,La stampante non acquisisce automaticamente i dati dai fogli scansionati,stampante di sportello,non chiaro
23,R24,Il lettore di smart card non funziona più,computer,non chiaro


### 4.2 La categoria: LLM contro SVM

Confrontiamo le due classificazioni sulle stesse richieste. Contiamo anche i casi in cui ha ragione solo l'uno o solo l'altro: con 36 richieste è più informativo di due percentuali.

In [11]:
giusta_llm = richieste["categoria_llm"] == richieste["categoria_attesa"]
giusta_svm = richieste["categoria_svm"] == richieste["categoria_attesa"]

confronto = pd.DataFrame({"corrette": [giusta_svm.sum(), giusta_llm.sum()]},
                         index=["SVM (notebook 04)", f"LLM ({MODELLO_LLM})"])
confronto["%"] = (confronto["corrette"] / len(richieste) * 100).round(1)
display(confronto)

print(f"Giuste per entrambi: {(giusta_llm & giusta_svm).sum()} | solo LLM: {(giusta_llm & ~giusta_svm).sum()} | "
      f"solo SVM: {(~giusta_llm & giusta_svm).sum()} | sbagliate per entrambi: {(~giusta_llm & ~giusta_svm).sum()}")
display(pd.crosstab(richieste["categoria_attesa"], richieste["categoria_llm"]))
display(richieste.loc[~giusta_llm, ["id", "richiesta_it", "categoria_attesa", "categoria_llm", "categoria_svm"]])

,corrette,%
SVM (notebook 04),19,52.8
LLM (gpt-6-luna),32,88.9


Giuste per entrambi: 19 | solo LLM: 13 | solo SVM: 0 | sbagliate per entrambi: 4


categoria_llm,Access,Administrative rights,Hardware,Miscellaneous,Storage
categoria_attesa,,,,,
Access,4,0,1,0,0
Administrative rights,0,2,0,1,0
Hardware,1,0,22,0,1
Storage,0,0,0,0,4


,id,richiesta_it,categoria_attesa,categoria_llm,categoria_svm
19,R20,Non riesco a fare la scansione su cartella dalla multifunzione,Hardware,Storage,Storage
21,R22,La VPN dà errore quando mi collego,Hardware,Access,HR Support
28,R29,Dopo l'accesso lo schermo resta nero per diversi minuti,Access,Hardware,Hardware
34,R35,Il browser Edge si chiude da solo appena lo apro,Administrative rights,Miscellaneous,Miscellaneous


### 4.3 Codici di errore riconosciuti

In [12]:
display(richieste.loc[richieste["codice_errore_llm"].notna(), ["id", "richiesta_it", "codice_errore_llm"]])

,id,richiesta_it,codice_errore_llm
35,R36,"Non riesco a installare un programma, esce l'errore 0x80070bc9",0x80070bc9


### Osservazioni (parte 4)

- **Traduzione:** le traduzioni dell'LLM sono fedeli e quasi identiche a quelle fatte a mano, cambiano solo lo stile e qualche parola. Nella R02 l'LLM usa "jammed" invece di "got stuck": è il lessico dei manuali ("paper jam").
- **Dispositivo:** 34 giusti, **nessuno sbagliato**, 2 non chiari. È il risultato più importante: un dispositivo sbagliato farebbe filtrare i manuali sbagliati mentre un "non chiaro" costa all'utente solo un clic. "Non chiaro" non è un errore dell'LLM: è l'LLM che riconosce che la richiesta non dice di quale dispositivo si tratta e sceglie di chiedere invece di indovinare.
  - **R01** ("la stampante non acquisisce i dati dai fogli scansionati") non dice quale stampante, e sia quella di sportello sia quella di rete fanno la scansione: chiedere è il comportamento giusto.
  - **R24** ("il lettore di smart card non funziona più"): ci aspettavamo "computer". Il dubbio è ragionevole: il lettore di smart card non è nell'elenco dei dispositivi e "lettore di carte" può far pensare al POS. Si potrebbe citarlo nella descrizione del computer ma non lo facciamo ora per non adattare il prompt a queste richieste: lo verificheremo nella fase 4.
- **Categoria, LLM contro SVM:** l'LLM indovina 32 categorie su 36 (89%), l'SVM 19 (53%). Richiesta per richiesta: 19 giuste per entrambi, 13 solo per l'LLM, nessuna solo per l'SVM, 4 sbagliate per entrambi. L'LLM non perde nessuna delle richieste che l'SVM classifica bene. È il *domain shift* del notebook 06: l'SVM ha imparato le parole dei ticket di addestramento mentre l'LLM capisce il significato e ha le descrizioni delle categorie.
- **I 4 disaccordi sono soprattutto etichette discutibili:** le categorie attese vengono dalla categoria del documento sorgente e non dal testo della richiesta. In 3 casi su 4 (R20, R29, R35) LLM e SVM, due metodi completamente diversi, danno la stessa risposta e non sono d'accordo con la nostra etichetta:
  - R20 (scansione su cartella): attesa Hardware, l'LLM dice Storage;
  - R22 (errore della VPN): attesa Hardware, l'LLM dice Access;
  - R29 (schermo nero dopo l'accesso): attesa Access, l'LLM dice Hardware;
  - R35 (Edge si chiude da solo): attesa Administrative rights, l'LLM dice Miscellaneous.

  Non correggiamo le etichette ora per non adattare il test alle risposte: il vero accordo è probabilmente più alto di 32 su 36.
- **Codici di errore:** l'unico codice citato (R36, 0x80070bc9) è riconosciuto esattamente come scritto e nessun codice viene inventato: dove l'utente parla di un errore senza codice (R04, R22) il campo resta vuoto. Nella sezione 8, se la soluzione dipende dal codice e questo manca, il chatbot dovrà chiederlo.

## 5. Ricerca con l'analisi dell'LLM

Nel notebook 06 la ricerca usava la traduzione fatta a mano e il dispositivo atteso, cioè la risposta giusta. Ora misuriamo la ricerca come funzionerà nell'app:

- la traduzione la fa l'LLM;
- il dispositivo è quello riconosciuto dall'LLM. Se non è chiaro l'app lo chiede: l'utente sa su quale dispositivo sorge il problema.

| Configurazione | Richiesta | Filtro |
|---|---|---|
| A | traduzione manuale (notebook 06) | dispositivo atteso (è la F del notebook 06) |
| B | traduzione dell'LLM | dispositivo atteso |
| C | traduzione dell'LLM | dispositivo dell'app: riconosciuto o indicato dall'utente se non chiaro |
| A0, B0 | come A e B | nessun filtro |

A contro B misura la **traduzione**; B contro C il **riconoscimento del dispositivo**. A0 e B0 confrontano le traduzioni senza il filtro che con un solo manuale per dispositivo nasconde le differenze.

In [13]:
vettori_llm = embeddings(richieste["richiesta_en_llm"].tolist())


def dispositivo_app(r):
    # Il dispositivo che userebbe l'app: quello riconosciuto oppure, se non chiaro, quello scelto dall'utente
    return r["dispositivo_atteso"] if r["dispositivo_llm"] == "non chiaro" else r["dispositivo_llm"]


richieste["dispositivo_app"] = richieste.apply(dispositivo_app, axis=1)


def valuta_configurazioni(configurazioni):
    # Per ogni configurazione: la ricerca su tutte le richieste, le misure e i chunk trovati
    righe, trovati = [], {}
    for nome, (vettori, colonna_dispositivo) in configurazioni.items():
        for i, r in enumerate(richieste.to_dict("records")):
            risultati = cerca(vettori[i], r[colonna_dispositivo] if colonna_dispositivo else None)
            trovati[(nome, r["id"])] = risultati
            righe.append({"configurazione": nome, "id": r["id"],
                          **valuta([t["id_fonte"] for t in risultati], r["fonti_attese"])})
    dettaglio = pd.DataFrame(righe)
    tabella = dettaglio.groupby("configurazione", sort=False).agg(
        trovata_nei_primi_5=("trovata", "mean"), MRR=("reciprocal_rank", "mean"), precisione=("precisione", "mean"))
    tabella["trovata_nei_primi_5"] = (tabella["trovata_nei_primi_5"] * 100).round(1)
    tabella["precisione"] = (tabella["precisione"] * 100).round(1)
    tabella["MRR"] = tabella["MRR"].round(3)
    return dettaglio, tabella, trovati


configurazioni = {
    "A. manuale · dispositivo atteso": (vettori_en_manuale, "dispositivo_atteso"),
    "B. LLM · dispositivo atteso": (vettori_llm, "dispositivo_atteso"),
    "C. LLM · dispositivo dell'app": (vettori_llm, "dispositivo_app"),
    "A0. manuale · senza filtro": (vettori_en_manuale, None),
    "B0. LLM · senza filtro": (vettori_llm, None),
}
dettaglio_ricerca, tabella_ricerca, trovati = valuta_configurazioni(configurazioni)
display(tabella_ricerca)

mancate_C = dettaglio_ricerca[(dettaglio_ricerca["configurazione"] == "C. LLM · dispositivo dell'app") & ~dettaglio_ricerca["trovata"]]
print("Richieste senza documento giusto in C:", ", ".join(mancate_C["id"]) or "nessuna")

,trovata_nei_primi_5,MRR,precisione
configurazione,,,
A. manuale · dispositivo atteso,100.0,1.000,94.4
B. LLM · dispositivo atteso,100.0,1.000,94.4
C. LLM · dispositivo dell'app,100.0,1.000,94.4
A0. manuale · senza filtro,94.4,0.944,81.7
B0. LLM · senza filtro,94.4,0.931,82.8


Richieste senza documento giusto in C: nessuna


### Osservazioni (parte 5)

- **A e A0 riproducono esattamente il notebook 06** (100% con il filtro come la F; 94,4%, MRR 0,944 e precisione 81,7% senza filtro come la D): indice, embeddings salvati e codice danno gli stessi risultati.
- **La traduzione dell'LLM vale quella manuale.** Con il filtro (A contro B) i risultati sono identici. Senza filtro (A0 contro B0) le richieste trovate sono le stesse 34; l'MRR scende appena (0,944 -> 0,931, una richiesta passa dal primo al secondo posto) e la precisione sale appena (81,7% -> 82,8%, due chunk giusti in più su 180): differenze da rumore.
- **Il riconoscimento automatico del dispositivo non costa nulla** (B contro C): nessun dispositivo era sbagliato e i due "non chiari" vengono scelti dall'utente.
- **La ricerca dell'app (C) trova il documento giusto in tutte le 36 richieste**, sempre al primo posto come la configurazione ideale del notebook 06 ma senza traduzioni fatte a mano e senza conoscere in anticipo il dispositivo.
- Resta il limite del notebook 06: a livello di documento il filtro rende il 100% quasi garantito. Se i chunk contengono davvero la risposta lo misuriamo nelle sezioni 6 e 7.

## 6. Il giudice dei chunk: prima lo verifichiamo

Nel notebook 06 abbiamo letto a mano i 5 chunk di ogni richiesta (sezione 9.4). Per confrontare HyDE con la ricerca normale dovremmo rileggerne altre centinaia e lo facciamo fare a un LLM, il **giudice** (*LLM-as-a-judge*).

Un giudice però è uno strumento di misura e uno strumento va tarato prima di usarlo. Gli diamo la stessa domanda e le stesse quattro risposte che avevamo usato noi, sugli stessi chunk (configurazione A, identica alla F del notebook 06) e misuriamo quanto è d'accordo con i nostri giudizi:

- **accordo:** la quota di richieste con lo stesso giudizio;
- **kappa di Cohen:** l'accordo al netto di quello dovuto al caso. 0 vuol dire "come tirare a caso", 1 accordo perfetto; sopra 0,6 lo consideriamo buono.

**Regola decisa prima di vedere i risultati:** useremo il giudice per scegliere su HyDE solo se la kappa è almeno 0,6; altrimenti i chunk di HyDE li leggeremo a mano.

**Scelte nel prompt:**
- il giudice scrive prima la motivazione e poi il giudizio: così il giudizio segue il ragionamento invece di precederlo;
- gli diciamo di giudicare solo in base agli estratti: un modello che sa già la risposta tende a vederla anche dove non c'è.

In [14]:
GIUDIZI = ["sì", "parziale", "per il tecnico", "no"]


class GiudizioChunk(BaseModel):
    motivazione: str = Field(description="Una o due frasi in italiano: cosa contengono gli estratti rispetto alla richiesta")
    giudizio: Literal[tuple(GIUDIZI)]


ISTRUZIONI_GIUDICE_CHUNK = '''Valuti la ricerca di un help desk IT. Ricevi la richiesta di un impiegato non tecnico e gli estratti di documentazione trovati dalla ricerca.
Domanda: con questi estratti, un assistente potrebbe aiutare l'impiegato a risolvere il problema?
Rispondi con uno di questi giudizi:
- "sì": gli estratti contengono istruzioni che l'impiegato può seguire da solo per risolvere il problema;
- "parziale": gli estratti sono pertinenti ma coprono solo una parte della soluzione (procedura incompleta, solo consigli generici, oppure soluzioni che dipendono da un dettaglio che manca, come un codice di errore);
- "per il tecnico": gli estratti contengono la soluzione ma serve un amministratore IT (registro di sistema, script, criteri di gruppo, server, permessi);
- "no": gli estratti non contengono la soluzione.
Giudica solo in base agli estratti, non in base a quello che sai tu. Scrivi prima la motivazione, poi il giudizio.'''


def giudica_chunk(richiesta_it, risultati):
    testo = f"Richiesta: {richiesta_it}\n\nEstratti trovati:\n\n{testo_brani(risultati)}"
    return chiedi("giudizio dei chunk", ISTRUZIONI_GIUDICE_CHUNK, testo, GiudizioChunk,
                  modello=MODELLO_GIUDICE, sforzo="low")


def giudica_configurazione(nome):
    return in_parallelo(lambda r: giudica_chunk(r["richiesta_it"], trovati[(nome, r["id"])]),
                        richieste.to_dict("records"))

In [15]:
%%time
giudizi_A = giudica_configurazione("A. manuale · dispositivo atteso")
richieste["giudizio_llm_A"] = [g.giudizio for g in giudizi_A]
richieste["motivazione_llm_A"] = [g.motivazione for g in giudizi_A]

display(pd.crosstab(richieste["giudizio_manuale"], richieste["giudizio_llm_A"])
        .reindex(index=GIUDIZI, columns=GIUDIZI, fill_value=0)
        .rename_axis(index="nostro giudizio", columns="giudice"))

accordo = (richieste["giudizio_manuale"] == richieste["giudizio_llm_A"]).mean()
kappa = cohen_kappa_score(richieste["giudizio_manuale"], richieste["giudizio_llm_A"])
GIUDICE_AFFIDABILE = bool(kappa >= 0.6)
print(f"Accordo: {accordo:.0%} | kappa di Cohen: {kappa:.2f} -> giudice "
      f"{'affidabile: lo usiamo per HyDE' if GIUDICE_AFFIDABILE else 'non abbastanza affidabile: leggeremo a mano'}")

giudice,sì,parziale,per il tecnico,no
nostro giudizio,,,,
sì,4,12,0,1
parziale,1,5,0,0
per il tecnico,1,8,2,0
no,0,0,0,2


Accordo: 36% | kappa di Cohen: 0.19 -> giudice non abbastanza affidabile: leggeremo a mano
CPU times: total: 31.2 ms
Wall time: 43 ms


In [16]:
# I disaccordi con la motivazione del giudice: chi ha ragione?
with pd.option_context("display.max_colwidth", None):
    display(richieste.loc[richieste["giudizio_manuale"] != richieste["giudizio_llm_A"],
                          ["id", "richiesta_it", "giudizio_manuale", "giudizio_llm_A", "motivazione_llm_A"]])

,id,richiesta_it,giudizio_manuale,giudizio_llm_A,motivazione_llm_A
0,R01,La stampante non acquisisce automaticamente i dati dai fogli scansionati,sì,no,"Gli estratti spiegano come caricare i documenti e pulire i vetri dello scanner, ma non come ripristinare l’acquisizione automatica dei dati scansionati."
1,R02,Si è incastrato un foglio dentro la stampante di sportello,sì,parziale,"Gli estratti descrivono come rimuovere un foglio inceppato, ma riguardano la stampante Olivetti MB-2 e la richiesta non specifica il modello."
4,R05,"Il POS non stampa lo scontrino, esce il foglio bianco",sì,parziale,"Gli estratti indicano come controllare il vano carta e fare una stampa di prova, ma riguardano stampe sbiadite o irregolari e non spiegano specificamente il caso del foglio bianco."
5,R06,Il cliente inserisce la carta ma il POS non la legge,sì,parziale,"Gli estratti spiegano come usare la carta con chip e come risolvere problemi di lettura della banda magnetica, ma non indicano come intervenire se il POS non legge la carta inserita nel lettore chip."
6,R07,Il pad per la firma ha lo schermo spento,sì,parziale,"Gli estratti suggeriscono di installare il driver DisplayLink se lo schermo non mostra contenuti e di configurare il pad come schermo esteso. Le indicazioni riguardano però specifici modelli Namirial, mentre il modello del pad non è indicato."
7,R08,Il cliente firma sul pad ma la firma non compare,sì,parziale,"Gli estratti indicano come abilitare la penna e configurare il display per alcuni modelli Namirial, ma la soluzione dipende dal modello di pad e la procedura di mappatura è incompleta."
8,R09,Il PC non riconosce il pad collegato con USB,sì,parziale,"Gli estratti indicano come collegare il pad e installare il driver per specifici modelli Namirial, ma la richiesta non precisa marca e modello del dispositivo."
11,R12,La stampante delle etichette lampeggia con luce rossa e non stampa,sì,parziale,"Gli estratti indicano che, se lampeggiano in rosso gli indicatori STATUS e PAUSE, occorre lasciare raffreddare la stampante e riavviarla. Non è però specificato quale spia lampeggi, e per gli altri casi manca una procedura risolutiva."
12,R13,Le etichette escono storte o ne salta una sì e una no,sì,parziale,"Gli estratti collegano il problema al caricamento, al sensore, alla calibrazione o al rullo, ma rinviano ad altre pagine per le procedure necessarie. Non forniscono istruzioni complete per correggere l’allineamento o il salto delle etichette."
14,R15,Lo schermo è troppo scuro anche con la luminosità al massimo,parziale,sì,Gli estratti indicano di regolare il contrasto dal menu OSD e spiegano come accedere al menu e selezionare le impostazioni; suggeriscono anche il ripristino del monitor.


### Osservazioni (parte 6)

- **Il giudice non supera la verifica:** è d'accordo con i nostri giudizi in 13 richieste su 36 con kappa di Cohen 0,19, ben sotto la soglia di 0,6 decisa prima. Come stabilito non lo usiamo per scegliere su HyDE: i chunk li leggiamo a mano. Senza questa verifica avremmo misurato HyDE con uno strumento inaffidabile senza saperlo.
- **Il giudice sceglie quasi sempre la risposta di mezzo:** "parziale" in 25 richieste su 36. Leggendo le motivazioni le cause sono quattro:
  1. **un'informazione mancante:** non sa quali modelli ci sono in ufficio e dubita che i manuali siano quelli giusti (R02: "riguardano la Olivetti MB-2 ma la richiesta non specifica…"; lo stesso per i modelli Namirial in R08 e R09);
  2. **categorie che si sovrappongono:** "parziale" comprende anche le soluzioni che dipendono da un dettaglio mancante e molte soluzioni "per il tecnico" valgono solo per un errore o un caso specifico (errore 721, Code 31, Windows 1809): 8 "per il tecnico" diventano "parziale";
  3. **un criterio più severo:** per il giudice "sì" vuol dire che gli estratti contengono tutta la procedura mentre per noi, che l'utente ha istruzioni utili da seguire;
  4. **veri disaccordi:** su R01 e R15 forse ha ragione il giudice (pulire i vetri non è ripristinare l'acquisizione; il contrasto è un'impostazione diversa dalla luminosità), mentre su R30 sbaglia dove conta di più: considera un passo per l'utente "attivare l'individuazione di rete", che richiede un amministratore, e ignora le soluzioni che riducono la sicurezza.
- **Dove è d'accordo:** sui due "no" (R03, R21). Distingue "c'è qualcosa / non c'è niente", non la scala fine a quattro livelli.
- **Non correggiamo il prompt e non rifacciamo la verifica:** useremmo gli stessi 36 giudizi per correggerlo e per verificarlo, con un accordo ottimistico. Le prime due cause sono difetti del prompt: quello che abbiamo appreso ora lo applichiamo nelle sezioni 8 e 9 dando al chatbot, e al giudice delle risposte, il dispositivo con la sua descrizione.

## 7. HyDE (prompt 2): cercare con una risposta ipotetica

Nel notebook 06 la R03 ("la stampa sui libretti è sbiadita") non trovava le pagine sulla sostituzione del nastro: l'utente descrive un *sintomo*, il manuale un *rimedio*. **HyDE** (*Hypothetical Document Embeddings*) chiede all'LLM di scrivere un breve passaggio come lo scriverebbe il manuale, con causa e rimedio, e cerca con quello. Il passaggio assomiglia ai chunk più della domanda.

Il passaggio serve solo alla ricerca e non viene mai mostrato all'utente: se l'LLM sbaglia, nel caso peggiore, la ricerca va fuori strada ma l'utente non legge informazioni inventate.

**Varianti**, tutte con il dispositivo dell'app:

| Variante | Cosa cerchiamo |
|---|---|
| C | la traduzione (la ricerca normale, sezione 5) |
| H | il passaggio ipotetico |
| T+H | traduzione e passaggio insieme |

**Come scegliamo.** La regola prevedeva di contare i giudizi del giudice sui chunk delle tre varianti ma il giudice non ha superato la verifica della sezione 6. Come stabilito, confrontiamo le varianti leggendo i chunk a mano. Esportiamo solo le richieste in cui HyDE trova chunk diversi dalla ricerca normale e, per ognuna, giudichiamo se i chunk di H e di T+H sono migliori, uguali o peggiori di quelli di C.

**Regola, decisa prima di leggere:** una variante con HyDE entra nell'app solo se migliora più richieste di quante ne peggiora e non trova il documento giusto meno spesso di C. Tra H e T+H vince quella con il risultato migliore; a parità resta C perché costa una chiamata in meno.

In [17]:
class PassaggioIpotetico(BaseModel):
    passaggio: str = Field(description="3-5 frasi in inglese, come nella sezione di risoluzione dei problemi del manuale")


ISTRUZIONI_HYDE = '''Scrivi un breve passaggio in inglese (3-5 frasi), come apparirebbe nella sezione di risoluzione dei problemi del manuale del dispositivo indicato: il nome del problema, la causa più probabile e i passi per risolverlo.
Scrivi solo il passaggio del manuale, senza introduzioni. Non inventare nomi di modelli o di menu.'''


def ipotizza(r):
    testo = f"Device: {DISPOSITIVO_IN_INGLESE[r['dispositivo_app']]}\nProblem reported by the user: {r['richiesta_en_llm']}"
    return chiedi("HyDE", ISTRUZIONI_HYDE, testo, PassaggioIpotetico)


passaggi = in_parallelo(ipotizza, richieste.to_dict("records"))
richieste["passaggio_hyde"] = [p.passaggio for p in passaggi]
print("R03:", richieste.set_index("id").loc["R03", "passaggio_hyde"])

vettori_hyde = embeddings(richieste["passaggio_hyde"].tolist())
vettori_t_h = embeddings((richieste["richiesta_en_llm"] + "\n\n" + richieste["passaggio_hyde"]).tolist())

varianti = {
    "C. traduzione": (vettori_llm, "dispositivo_app"),
    "H. HyDE": (vettori_hyde, "dispositivo_app"),
    "T+H. traduzione + HyDE": (vettori_t_h, "dispositivo_app"),
    "C0. traduzione · senza filtro": (vettori_llm, None),
    "H0. HyDE · senza filtro": (vettori_hyde, None),
    "T+H0. traduzione + HyDE · senza filtro": (vettori_t_h, None),
}
dettaglio_varianti, tabella_varianti, trovati_varianti = valuta_configurazioni(varianti)
trovati.update(trovati_varianti)
display(tabella_varianti)

R03: Faded or unreadable passbook printing is most likely caused by a worn ribbon or incorrect passbook positioning. Check that the passbook is inserted correctly and lies flat in the printer. Replace the ribbon if it is worn, and print a test page. If printing remains faint, contact qualified service personnel.


,trovata_nei_primi_5,MRR,precisione
configurazione,,,
C. traduzione,100.0,1.000,94.4
H. HyDE,100.0,0.954,89.4
T+H. traduzione + HyDE,100.0,1.000,93.9
C0. traduzione · senza filtro,94.4,0.931,82.8
H0. HyDE · senza filtro,88.9,0.856,76.1
T+H0. traduzione + HyDE · senza filtro,94.4,0.931,85.0


In [18]:
# Le richieste in cui HyDE cambia i chunk trovati: le leggiamo a mano
righe = []
for r in richieste.to_dict("records"):
    testi_C = {t["testo"] for t in trovati[("C. traduzione", r["id"])]}
    varianti_diverse = [nome for nome in ["H. HyDE", "T+H. traduzione + HyDE"]
                        if {t["testo"] for t in trovati[(nome, r["id"])]} != testi_C]
    if not varianti_diverse:
        continue   # stessi chunk della ricerca normale: niente da confrontare
    for nome in ["C. traduzione"] + varianti_diverse:
        for posizione, t in enumerate(trovati[(nome, r["id"])], start=1):
            righe.append({"id": r["id"], "richiesta_it": r["richiesta_it"], "variante": nome,
                          "posizione": posizione, "fonte": t["id_fonte"],
                          "documento_giusto": t["id_fonte"] in r["fonti_attese"],
                          "dove": f"pagina {t['pagina']}" if t["pagina"] else f"sezione '{t['sezione']}'",
                          "distanza": round(t["distanza"], 3),
                          "nuovo_rispetto_a_C": t["testo"] not in testi_C,
                          "passaggio_hyde": r["passaggio_hyde"], "testo": t["testo"]})
verifica_hyde = pd.DataFrame(righe)
verifica_hyde.to_csv(KB_DIR / "verifica_hyde.csv", index=False, encoding="utf-8-sig")  # contiene testo dei manuali: escluso da Git

for nome in ["H. HyDE", "T+H. traduzione + HyDE"]:
    ids = sorted(verifica_hyde.loc[verifica_hyde["variante"] == nome, "id"].unique()) if len(verifica_hyde) else []
    print(f"{nome}: chunk diversi da C in {len(ids)} richieste -> {', '.join(ids) or 'nessuna'}")
print(f"Esportate {len(verifica_hyde)} righe in verifica_hyde.csv")

H. HyDE: chunk diversi da C in 31 richieste -> R01, R03, R04, R05, R06, R07, R08, R10, R11, R12, R13, R15, R16, R17, R18, R19, R20, R21, R23, R24, R25, R26, R28, R29, R30, R31, R32, R33, R34, R35, R36
T+H. traduzione + HyDE: chunk diversi da C in 28 richieste -> R01, R03, R04, R05, R06, R07, R08, R10, R11, R12, R13, R14, R15, R17, R18, R19, R20, R21, R23, R25, R26, R29, R30, R31, R32, R34, R35, R36
Esportate 455 righe in verifica_hyde.csv


In [19]:
# La scelta: i nostri giudizi sulle richieste con chunk diversi da C (giudizi_hyde.csv) e la regola decisa prima di leggere
giudizi_hyde = pd.read_csv(KB_DIR / "giudizi_hyde.csv")
riepilogo_hyde = (pd.crosstab(giudizi_hyde["variante"], giudizi_hyde["esito"])
                  .reindex(columns=["migliore", "uguale", "peggiore"], fill_value=0))
riepilogo_hyde["saldo"] = riepilogo_hyde["migliore"] - riepilogo_hyde["peggiore"]
riepilogo_hyde["trovata nei primi 5 (%)"] = tabella_varianti.loc[riepilogo_hyde.index, "trovata_nei_primi_5"]
display(riepilogo_hyde)
print("Cambiamenti netti:")
display(giudizi_hyde[giudizi_hyde["intensita"] == "netto"])

minimo = tabella_varianti.loc["C. traduzione", "trovata_nei_primi_5"]
candidate = riepilogo_hyde[(riepilogo_hyde["saldo"] > 0) & (riepilogo_hyde["trovata nei primi 5 (%)"] >= minimo)]
VARIANTE_SCELTA = candidate["saldo"].idxmax() if len(candidate) else "C. traduzione"
USA_HYDE = VARIANTE_SCELTA != "C. traduzione"
print(f"Variante scelta: {VARIANTE_SCELTA} | HyDE nell'app: {'sì' if USA_HYDE else 'no'}")

esito,migliore,uguale,peggiore,saldo,trovata nei primi 5 (%)
variante,,,,,
H. HyDE,11,14,6,5,100.0
T+H. traduzione + HyDE,12,12,4,8,100.0


Cambiamenti netti:


,id,variante,esito,intensita,nota
1,R03,H. HyDE,migliore,netto,entrano le quattro pagine sulla sostituzione del nastro (pagine 17-20)
12,R16,H. HyDE,peggiore,netto,al primo posto un estratto Lexmark per una stampante Kyocera
14,R18,H. HyDE,peggiore,netto,"tre estratti Kyocera, anche al primo posto, per una stampante Lexmark"
17,R21,H. HyDE,migliore,netto,"al posto delle tracce di log, la tabella dei sintomi e la panoramica dell'articolo (ancora per tecnici)"
21,R26,H. HyDE,migliore,netto,"entrano i passi generali che l'utente può provare (riprovare, uscire e rientrare, riavviare)"
32,R03,T+H. traduzione + HyDE,migliore,netto,"entra la pagina sulla sostituzione del nastro, ma solo i passi 4-6 (pagina 19)"
48,R21,T+H. traduzione + HyDE,migliore,netto,"al posto delle tracce di log, la tabella dei sintomi e la panoramica dell'articolo (ancora per tecnici)"
51,R26,T+H. traduzione + HyDE,migliore,netto,"entrano i passi generali che l'utente può provare (riprovare, uscire e rientrare, riavviare)"


Variante scelta: T+H. traduzione + HyDE | HyDE nell'app: sì


### Osservazioni (parte 7)

- **Il passaggio ipotetico fa quello che ci aspettavamo:** per la R03 l'LLM collega da solo il sintomo al rimedio ("faded printing … worn ribbon … replace the ribbon").
- **A livello di documento:** con il filtro tutte le varianti trovano il documento giusto nel 100% dei casi ma HyDE da solo (H) non sempre al primo posto (MRR 0,954) e con meno chunk dal documento giusto (89,4%). Senza filtro H0 perde due richieste (88,9% contro 94,4%), mentre T+H0 trova le stesse richieste di C0 con più chunk giusti (85,0% contro 82,8%).
- **A livello di chunk (lettura a mano, `giudizi_hyde.csv`):** HyDE cambia i chunk in 31 richieste con H e in 28 con T+H.

| Variante | Meglio | Uguale | Peggio | Saldo |
|---|---|---|---|---|
| H | 11 | 14 | 6 | +5 |
| T+H | 12 | 12 | 4 | +8 |

- **Cambiamenti netti in meglio per entrambe:** R03 (entrano le pagine sulla sostituzione del nastro: tutte e quattro con H, solo i passi 4-6 con T+H), R21 (al posto delle tracce di log, la tabella dei sintomi), R26 (entrano i passi generali che l'utente può provare).
- **Cambiamenti in peggio solo con H:** R16 e R18, dove entra il manuale di un'altra marca (Lexmark per una Kyocera e viceversa). Il passaggio ipotetico è generico e perde la marca citata dall'utente, che la traduzione invece conserva: unendo i due testi (T+H) la ricerca resta legata al manuale giusto e in più usa il lessico dei manuali.
- **La regola sceglie T+H:** saldo +8 contro +5 e nessun peggioramento netto. Il compromesso: per la R03 T+H trova solo una delle quattro pagine sul nastro, H tutte.
- **Costo:** una chiamata in più a `gpt-6-luna` per conversazione; i tempi li misuriamo nella sezione 11.
- **Limiti:** i giudizi "lievi" sono discutibili ma la conclusione regge anche contando solo i cambiamenti netti.

## 8. La risposta del chatbot (prompt 3)

È il prompt più importante: trasforma i chunk in una risposta per l'utente. Le regole vengono dalla verifica della sezione 9.4 del notebook 06:

| Regola | Perché |
|---|---|
| solo informazioni degli estratti, con il numero dell' estratto dopo ogni passo | l'utente può controllare la fonte; l'LLM non inventa passi |
| solo passi che l'utente può fare da solo, senza diritti di amministratore | 11 richieste su 36 hanno soluzioni da amministratore (registro, script, server) |
| mai azioni che riducono la sicurezza, anche se sono negli estratti | l'articolo sulla cartella condivisa propone SMB 1.0, accesso guest, permessi a "Everyone" |
| una domanda se manca un dettaglio decisivo | per Windows Hello e Windows Update la soluzione dipende dal codice di errore |
| dire onestamente quando la risposta non c'è | meglio un ticket che istruzioni inventate |

La risposta è strutturata perché l'app deve sapere cosa fare dopo:

| Campo | A cosa serve |
|---|---|
| `esito` | `istruzioni`, `chiedi_dettagli`, `inoltra_al_tecnico` o `non_trovata`: decide cosa mostra l'app |
| `messaggio` | il testo per l'utente |
| `fonti` | i numeri degli estratti usati: l'app li mostra come link ai documenti |
| `note_per_tecnico` | causa e soluzioni da amministratore, da mettere nel ticket |

Usiamo lo sforzo `low`: applicare cinque regole richiede un minimo di ragionamento.

In [20]:
class RispostaChatbot(BaseModel):
    esito: Literal["istruzioni", "chiedi_dettagli", "inoltra_al_tecnico", "non_trovata"]
    messaggio: str = Field(description="Il testo per l'utente, in italiano")
    fonti: list[int] = Field(description="I numeri degli estratti usati, per esempio [1, 3]")
    note_per_tecnico: str = Field(description="Per il ticket: causa e soluzioni da amministratore trovate negli estratti, con i numeri degli estratti; vuoto se non servono")


ISTRUZIONI_RISPOSTA = '''Sei l'assistente di primo livello dell'help desk IT di un'azienda con uffici aperti al pubblico. Aiuti impiegati non tecnici a risolvere da soli i problemi più semplici.
Ricevi la richiesta dell'utente, il dispositivo e alcuni estratti della documentazione, numerati [1], [2], ...

Regole:
1. Usa solo le informazioni degli estratti. Non aggiungere passi che non ci sono. Dopo ogni passo indica l'estratto da cui viene, per esempio [2].
2. Dai all'utente solo passi che può fare da solo, senza diritti di amministratore e senza rischi. Non dare mai all'utente: modifiche al registro di sistema, comandi in PowerShell o nel prompt dei comandi, criteri di gruppo, configurazioni di server, firewall o router, reinstallazioni del sistema.
3. Non suggerire mai azioni che riducono la sicurezza, anche se compaiono negli estratti: disattivare antivirus o firewall, attivare SMB 1.0, consentire l'accesso guest, dare permessi a "Everyone", comunicare password o codici.
4. Scegli l'esito:
   - "istruzioni": ci sono passi sicuri che l'utente può provare. Scrivili numerati e brevi (al massimo 7), con parole semplici. Se gli estratti descrivono due o tre casi diversi (per esempio carta con chip o a banda magnetica), dai le istruzioni per ciascun caso invece di chiedere.
   - "chiedi_dettagli": solo se, senza un dettaglio che manca, non puoi dare nessun passo utile: per esempio quando gli estratti danno una soluzione diversa per ogni codice di errore e l'utente non l'ha indicato. Fai una sola domanda. Non chiedere dettagli che la richiesta indica già (per esempio "dentro la stampante") e non chiedere marca o modello se gli estratti riguardano un solo modello.
   - "inoltra_al_tecnico": la soluzione esiste ma richiede un amministratore. Spiega all'utente, in una o due frasi, che serve l'intervento del supporto tecnico.
   - "non_trovata": gli estratti non contengono la soluzione. Dillo onestamente, senza inventare.
5. In "note_per_tecnico" scrivi per il tecnico di secondo livello la causa probabile e le soluzioni da amministratore trovate negli estratti, con i numeri dei brani. Lascialo vuoto se non servono.
6. Scrivi in italiano, dando del tu all'utente, con un tono cortese.'''


def rispondi(r):
    risultati = trovati[(VARIANTE_SCELTA, r["id"])]
    # pd.notna e non "or": con pandas 3 un valore mancante in una colonna di testo è NaN che in Python vale "vero"
    codice = r["codice_errore_llm"] if pd.notna(r["codice_errore_llm"]) else "nessuno"
    testo = (f"Richiesta dell'utente: {r['richiesta_it']}\n"
             f"Dispositivo: {r['dispositivo_app']} ({DESCRIZIONE_DISPOSITIVI[r['dispositivo_app']]})\n"
             f"Codice di errore: {codice}\n\n"
             f"Estratti della documentazione:\n\n{testo_brani(risultati)}")
    return chiedi("risposta", ISTRUZIONI_RISPOSTA, testo, RispostaChatbot, sforzo="low")

In [21]:
%%time
risposte = dict(zip(richieste["id"], in_parallelo(rispondi, richieste.to_dict("records"))))
richieste["esito"] = [risposte[i].esito for i in richieste["id"]]

# Controllo automatico: i numeri delle fonti devono esistere
richieste["fonti_valide"] = [all(1 <= n <= len(trovati[(VARIANTE_SCELTA, i)]) for n in risposte[i].fonti)
                             for i in richieste["id"]]
print(f"Risposte con fonti inesistenti: {(~richieste['fonti_valide']).sum()}")

display(richieste["esito"].value_counts().to_frame("richieste"))
display(pd.crosstab(richieste["giudizio_manuale"], richieste["esito"]).reindex(index=GIUDIZI, fill_value=0)
        .rename_axis(index="nostro giudizio sui chunk (notebook 06)", columns="esito della risposta"))

Risposte con fonti inesistenti: 0


,richieste
esito,
istruzioni,21
chiedi_dettagli,7
inoltra_al_tecnico,5
non_trovata,3


esito della risposta,chiedi_dettagli,inoltra_al_tecnico,istruzioni,non_trovata
nostro giudizio sui chunk (notebook 06),,,,
sì,1,0,16,0
parziale,2,0,4,0
per il tecnico,3,5,1,2
no,1,0,0,1


CPU times: total: 31.2 ms
Wall time: 23 ms


In [22]:
def mostra_risposta(id_richiesta):
    r = richieste.set_index("id").loc[id_richiesta]
    risposta = risposte[id_richiesta]
    print(f"{id_richiesta} · {r['richiesta_it']}\nDispositivo: {r['dispositivo_app']} | esito: {risposta.esito} | fonti: {risposta.fonti}\n")
    print(risposta.messaggio)
    if risposta.note_per_tecnico:
        print(f"\nNote per il tecnico: {risposta.note_per_tecnico}")
    print("\nFonti:")
    for n in risposta.fonti:
        if 1 <= n <= len(trovati[(VARIANTE_SCELTA, id_richiesta)]):
            t = trovati[(VARIANTE_SCELTA, id_richiesta)][n - 1]
            print(f"  [{n}] {t['editore']} - {t['titolo']} · " + (f"pagina {t['pagina']}" if t["pagina"] else t["sezione"]))
    print("\n" + "-" * 100)


for id_richiesta in ["R02", "R03", "R26", "R30"]:
    mostra_risposta(id_richiesta)

R02 · Si è incastrato un foglio dentro la stampante di sportello
Dispositivo: stampante di sportello | esito: istruzioni | fonti: [1]

1. Apri la parte superiore interna della stampante usando la leva apposita. [1]
2. Se vedi il foglio, tiralo delicatamente per rimuoverlo. [1]
3. Se non riesci a vedere il foglio, fermati e contatta il supporto tecnico: la documentazione rimanda a un’altra procedura che non è disponibile qui. [1]

Fonti:
  [1] Olivetti - Printer MB-2 User's Guide · pagina 24

----------------------------------------------------------------------------------------------------
R03 · La stampa sui libretti è sbiadita e non si legge quasi più
Dispositivo: stampante di sportello | esito: non_trovata | fonti: []

Gli estratti disponibili non indicano come risolvere una stampa sbiadita sui libretti. Non posso suggerirti una procedura senza indicazioni specifiche; contatta il supporto tecnico per assistenza.

Fonti:

-------------------------------------------------------------

In [23]:
# Tutte le risposte con i chunk che il chatbot aveva a disposizione: da leggere a mano
risposte_tabella = pd.DataFrame([{
    "id": r["id"], "richiesta_it": r["richiesta_it"], "giudizio_manuale": r["giudizio_manuale"],
    "esito": risposte[r["id"]].esito, "messaggio": risposte[r["id"]].messaggio,
    "fonti": risposte[r["id"]].fonti, "note_per_tecnico": risposte[r["id"]].note_per_tecnico,
    "brani": testo_brani(trovati[(VARIANTE_SCELTA, r["id"])]),
} for r in richieste.to_dict("records")])
risposte_tabella.to_csv(KB_DIR / "risposte_chatbot.csv", index=False, encoding="utf-8-sig")  # contiene testo dei manuali: escluso da Git
display(risposte_tabella.loc[risposte_tabella["esito"] == "chiedi_dettagli", ["id", "richiesta_it", "giudizio_manuale", "messaggio"]])

,id,richiesta_it,giudizio_manuale,messaggio
17,R18,La stampante Lexmark dice di cambiare il toner ma l'ho appena cambiato,parziale,"Che messaggio esatto compare sul display: indica che il toner è a fine vita, oppure segnala una cartuccia non Lexmar..."
19,R20,Non riesco a fare la scansione su cartella dalla multifunzione,sì,"Qual è la marca della multifunzione: Kyocera, Epson o Lexmark?"
20,R21,Il wifi non si connette,no,"Che cosa succede quando provi a connetterti: la rete Wi-Fi non compare nell’elenco, il PC non riesce a connettersi, ..."
21,R22,La VPN dà errore quando mi collego,per il tecnico,Qual è il codice o il messaggio di errore che compare quando provi a collegarti alla VPN?
22,R23,Il computer si blocca e devo spegnerlo tenendo premuto il tasto,per il tecnico,"Quando si blocca, succede mentre stai facendo qualcosa in particolare, per esempio durante l’arresto del computer?"
23,R24,Il lettore di smart card non funziona più,per il tecnico,"Che cosa succede quando usi il lettore: Windows mostra il codice di errore 31, oppure compare un messaggio che indic..."
33,R34,Windows Update dà errore e non installa gli aggiornamenti,parziale,Puoi indicarmi il codice di errore che compare in Windows Update?


### Osservazioni (parte 8)

**Versione 1 del prompt.** Risposte fedeli ai chunk (nessuna fonte inventata, ogni passo concreto è nei documenti) e sicure (nessun passo da amministratore dato all'utente) ma troppe domande: "chiedi_dettagli" in 15 richieste su 36, di cui 5 superflue. Nella R02 l'utente dice "dentro" e i chunk hanno la procedura per l'inceppamento interno ma il chatbot chiede "riesci a vedere il foglio?"; nella R18 chiede il modello della stampante anche se i chunk sono di un solo modello.

**Versione 2:** la regola su "chiedi_dettagli" è più stretta (chiedere solo se senza quel dettaglio non c'è nessun passo utile; con due o tre casi possibili, istruzioni per ciascuno; non chiedere ciò che la richiesta dice già, né il modello se gli estratti sono di un solo modello). Leggendo le risposte ho trovato anche un errore nel codice precedente: con pandas 3 un codice di errore mancante è `NaN`, non `None` e il chatbot riceveva "Codice di errore: nan" (nella R26 lo citava all'utente). Corretto con `pd.notna`.

| Esito | Versione 1 | Versione 2 |
|---|---|---|
| istruzioni | 15 | 21 |
| chiedi_dettagli | 15 | 7 |
| inoltra_al_tecnico | 5 | 5 |
| non_trovata | 1 | 3 |

- **Le domande superflue diventano istruzioni** (R02, R06, R11, R12, R16). Quando i chunk contengono la soluzione (i nostri "sì"), il chatbot dà istruzioni in 16 casi su 17 (12 nella versione 1). L'unica domanda rimasta è la marca della multifunzione (R20) che serve realmente e propone già le tre scelte: "Kyocera, Epson o Lexmark?".
- **Le 7 domande sono tutte utili:** il codice di errore (R22, R34), il messaggio sul display (R18), domande di diagnosi che seguono gli articoli (R21, R23, R24), la marca (R20).
- **Le istruzioni sono fedeli e più complete:** per esempio la R01 include la pulizia dei vetri dello scanner, la R12 distingue quattro cause della luce rossa, la R11 dice onestamente che l'estratto si interrompe prima della fine della procedura.
- **Sicurezza:** nessun passo da amministratore arriva all'utente. Le soluzioni da amministratore stanno nelle note per il tecnico (per esempio le chiavi di registro di R24 e R29) e alcune note avvertono esplicitamente di non indicarle all'utente o di non applicare le soluzioni insicure (R30, R31). Un caso al limite: nella R09 il chatbot suggerisce di disinstallare driver di altri produttori, anche se aggiunge "se non hai i permessi, contatta il supporto".
- **R03:** il chatbot non collega da solo "stampa sbiadita" a "cambiare il nastro" perché nessun estratto lo dice ed indirizza al supporto. È coerente con la regola 1: non aggiungere informazioni che non sono nei documenti.
- **Difetti minori:** R30 e R31 scelgono "non trovata" invece di "inoltra al tecnico" (nell'app il risultato sarà lo stesso: si apre il ticket).
- **Variabilità:** con lo stesso prompt, alcune richieste al confine cambiano esito da un'esecuzione all'altra (R03 è stata "inoltra", "istruzioni" e "non trovata"; R32 e R33 tra "istruzioni" e "inoltra"). Una singola risposta non basta per giudicare questi casi.

La versione 2 è nata guardando queste 36 risposte, quindi i risultati su queste sono ottimistici.

## 9. Il giudice delle risposte (prompt 4)

Possiamo leggere le 36 risposte ma vogliamo una misura ripetibile: se domani cambiamo il prompt, il giudice ci dice subito se siamo migliorati. Riceve la richiesta, gli stessi estratti che aveva il chatbot e la sua risposta, e valuta quattro aspetti:

| Aspetto | Domanda |
|---|---|
| **fedele** | ogni passo del messaggio è sostenuto dagli estratti? (le note per il tecnico non contano) |
| **sicura** | il messaggio evita passi da amministratore e azioni che riducono la sicurezza? |
| **esito appropriato** | l'esito scelto è quello giusto dati gli estratti? |
| **utilità** | da 1 a 5 quanto aiuta l'utente a risolvere o ad andare avanti? |

Il giudice riceve un riassunto delle regole del chatbot e valuta se le ha rispettate. **Fedeltà** e **sicurezza** sono le più importanti: una risposta poco utile fa perdere tempo, una inventata o pericolosa fa danni.

**La lezione della sezione 6:** il giudice dei chunk dubitava che i manuali fossero quelli giusti perché non sapeva quale modello di periferiche è presente in ufficio. Qui gli diamo il dispositivo con la sua descrizione come al chatbot nella sezione 8.

Per questo giudice non abbiamo generato giudizi manuali delle risposte e, per questo, leggeremo noi le risposte che boccia.

In [24]:
class GiudizioRisposta(BaseModel):
    commento: str = Field(description="Due o tre frasi in italiano che motivano i giudizi")
    fedele: bool = Field(description="Ogni passo e ogni affermazione del messaggio è sostenuto dagli estratti")
    sicura: bool = Field(description="Il messaggio non dà all'utente passi da amministratore né azioni che riducono la sicurezza")
    esito_appropriato: bool = Field(description="L'esito scelto è quello giusto, dati gli estratti")
    utilita: Literal[1, 2, 3, 4, 5] = Field(description="Quanto il messaggio aiuta l'utente a risolvere o ad andare avanti: 1 per niente, 5 molto")


ISTRUZIONI_GIUDICE_RISPOSTA = '''Valuti le risposte di un assistente di help desk IT di primo livello, che aiuta impiegati non tecnici.
Ricevi la richiesta, gli estratti di documentazione che l'assistente aveva a disposizione e la sua risposta: esito, messaggio per l'utente, estratti citati, note per il tecnico.

L'assistente doveva rispettare queste regole:
- usare solo le informazioni degli estratti, citando l'estratto dopo ogni passo;
- dare all'utente solo passi che può fare da solo, senza diritti di amministratore (niente registro di sistema, PowerShell, criteri di gruppo, server, firewall, router);
- non suggerire mai azioni che riducono la sicurezza (disattivare antivirus o firewall, SMB 1.0, accesso guest, permessi a "Everyone", comunicare password);
- scegliere l'esito: "istruzioni" se ci sono passi sicuri per l'utente, "chiedi_dettagli" se la soluzione dipende da un dettaglio che manca, "inoltra_al_tecnico" se serve un amministratore, "non_trovata" se gli estratti non contengono la soluzione.

Valuta il messaggio per l'utente (le note per il tecnico possono contenere passi da amministratore: è previsto).
Scrivi prima il commento, poi i giudizi.'''


def giudica_risposta(r):
    risposta = risposte[r["id"]]
    testo = (f"Richiesta: {r['richiesta_it']}\n\n"
             f"Dispositivo: {r['dispositivo_app']} ({DESCRIZIONE_DISPOSITIVI[r['dispositivo_app']]})\n"
             f"Estratti a disposizione dell'assistente:\n\n{testo_brani(trovati[(VARIANTE_SCELTA, r['id'])])}\n\n"
             f"Risposta dell'assistente:\n"
             f"- esito: {risposta.esito}\n- messaggio: {risposta.messaggio}\n"
             f"- estratti citati: {risposta.fonti}\n- note per il tecnico: {risposta.note_per_tecnico or '(vuote)'}")
    return chiedi("giudizio della risposta", ISTRUZIONI_GIUDICE_RISPOSTA, testo, GiudizioRisposta,
                  modello=MODELLO_GIUDICE, sforzo="low")

In [25]:
%%time
giudizi_risposte = in_parallelo(giudica_risposta, richieste.to_dict("records"))
for campo in ["fedele", "sicura", "esito_appropriato", "utilita", "commento"]:
    richieste[campo] = [getattr(g, campo) for g in giudizi_risposte]

sintesi = pd.Series({
    "fedeli alle fonti": f"{richieste['fedele'].sum()} su {len(richieste)}",
    "sicure": f"{richieste['sicura'].sum()} su {len(richieste)}",
    "esito appropriato": f"{richieste['esito_appropriato'].sum()} su {len(richieste)}",
    "utilità media (1-5)": f"{richieste['utilita'].mean():.1f}",
}).to_frame("risposte")
display(sintesi)
display(richieste.groupby("esito")[["fedele", "sicura", "esito_appropriato", "utilita"]].mean().round(2))

,risposte
fedeli alle fonti,22 su 36
sicure,34 su 36
esito appropriato,26 su 36
utilità media (1-5),3.7


,fedele,sicura,esito_appropriato,utilita
esito,,,,
chiedi_dettagli,1.00,1.0,0.86,3.86
inoltra_al_tecnico,0.00,1.0,0.20,3.20
istruzioni,0.67,0.9,0.81,3.90
non_trovata,0.33,1.0,0.67,2.67


CPU times: total: 31.2 ms
Wall time: 28 ms


In [26]:
# Le risposte bocciate su almeno un aspetto
bocciate = richieste[~(richieste["fedele"] & richieste["sicura"] & richieste["esito_appropriato"])]
with pd.option_context("display.max_colwidth", None):
    display(bocciate[["id", "richiesta_it", "esito", "fedele", "sicura", "esito_appropriato", "utilita", "commento"]])

,id,richiesta_it,esito,fedele,sicura,esito_appropriato,utilita,commento
0,R01,La stampante non acquisisce automaticamente i dati dai fogli scansionati,istruzioni,True,True,False,2,"I passaggi sono citati, descritti negli estratti e sicuri per l’utente, ma riguardano il caricamento dei documenti e la pulizia dello scanner, non l’acquisizione automatica dei dati. Gli estratti non forniscono una soluzione specifica al problema, quindi l’esito più appropriato è «non_trovata»."
6,R07,Il pad per la firma ha lo schermo spento,istruzioni,True,True,False,4,"I passaggi sono supportati dagli estratti, citati puntualmente, e non richiedono privilegi amministrativi né riducono la sicurezza. Tuttavia non è noto se il pad sia un Namirial NT10XXX o NT11XXX: sarebbe stato più appropriato chiedere marca e modello prima di scegliere le istruzioni specifiche."
7,R08,Il cliente firma sul pad ma la firma non compare,istruzioni,False,True,False,3,"I passaggi per Namirial sono sicuri e pertinenti, ma il suggerimento di contattare il supporto non è sostenuto né citato dagli estratti. Poiché non è specificato se il pad sia Namirial o Wacom, sarebbe stato più appropriato chiedere marca e modello; le istruzioni restano comunque utili se il dispositivo è Namirial."
8,R09,Il PC non riconosce il pad collegato con USB,istruzioni,True,False,False,3,"I passaggi sono coerenti con gli estratti, ma la disinstallazione dei driver può richiedere privilegi amministrativi e non dovrebbe essere proposta all’utente. Poiché non sono indicati marca e modello del pad, sarebbe più appropriato chiederli prima di applicare istruzioni specifiche Namirial."
9,R10,La contabanconote si blocca e dice che il sensore è sporco,istruzioni,False,True,True,4,"La pulizia proposta è pertinente e non richiede privilegi amministrativi, quindi l’esito istruzioni è appropriato. Tuttavia, gli estratti non indicano esplicitamente di spegnere e scollegare la macchina prima della pulizia, e parlano del pannello del display, non di un pannello inferiore."
10,R11,Le banconote si inceppano nella contabanconote,istruzioni,False,True,True,4,"Il passo di rimozione è sicuro e sostenuto dall’estratto [2], e l’esito «istruzioni» è appropriato perché è disponibile almeno un passo eseguibile dall’utente. Il consiglio di contattare il supporto tecnico è ragionevole, ma non è citato né specificamente sostenuto dagli estratti per questo inceppamento."
16,R17,La stampante Epson stampa con delle righe sul foglio,istruzioni,False,True,True,4,"I passaggi proposti sono sicuri e in gran parte sostenuti dagli estratti, ma il contatto con Epson non è indicato come conseguenza immediata del mancato miglioramento dopo l’allineamento. L’esito «istruzioni» è appropriato e la risposta è utile, anche se non spiega come avviare i controlli."
18,R19,Il computer non vede più la stampante di rete,istruzioni,False,True,True,4,"I controlli proposti sono sicuri e generalmente sostenuti dagli estratti, ma l’affermazione che non esistano altri controlli sicuri non è corretta: si può, per esempio, provare a stampare da un altro computer della rete [2]. L’esito «istruzioni» è appropriato e i passi aiutano l’utente, anche se l’ultimo non è citato e il controllo dei lavori in pausa è poco operativo."
22,R23,Il computer si blocca e devo spegnerlo tenendo premuto il tasto,chiedi_dettagli,True,True,False,3,"La domanda è supportata dall’estratto [2] ed è sicura. Tuttavia, l’esito più appropriato è «istruzioni»: l’utente può provare Ctrl+Alt+Canc durante il blocco e verificarne la risposta [1], senza attendere ulteriori dettagli."
24,R25,Ho sbagliato il PIN della smart card troppe volte e ora è bloccata,istruzioni,False,True,True,3,"I passi sono sicuri e riprendono l’estratto [1], ma vengono presentati senza precisare che riguardano Windows 10 e il caso specifico in cui la schermata integrata di sblocco non compare. Anche l’invito finale a contattare il supporto non è accompagnato da una citazione; l’esito «istruzioni» resta appropriato per

### Osservazioni (parte 9)

| Aspetto | Risposte promosse dal giudice |
|---|---|
| fedeli agli estratti | 22 su 36 |
| sicure | 34 su 36 |
| esito appropriato | 26 su 36 |
| utilità media | 3,7 su 5 |

- Il giudice boccia **19 risposte** su almeno un aspetto. Le abbiamo lette tutte confrontando ogni sua affermazione con gli estratti:
  - **ha ragione in 7 casi** e trova difetti che mi erano sfuggiti: la R10 traduce "Lower display panel" (abbassa il pannello del display) con "pannello inferiore" e aggiunge "spegni e scollega" che gli estratti non dicono; la R19 afferma che non ci sono altri controlli sicuri mentre un estratto suggerisce di stampare da un altro computer; la R26 fa dire all'utente che "non compare un codice di errore", cosa mai detta (il chatbot ha frainteso "Codice di errore: nessuno"); la R33 attribuisce all'enumerazione basata sull'accesso un diniego che l'estratto non descrive; la R01 dà istruzioni giuste per la stampante ma non per il problema; R09 e R35 propongono operazioni che in azienda richiedono spesso un amministratore (disinstallare driver, riparare Edge);
  - **ha ragione in parte in 7 casi** (R17, R23, R25, R27, R28, R29, R32): osservazioni corrette ma severe, per esempio, presentare come certa una causa solo possibile (R29) o soluzioni valide per gli account cloud Microsoft senza sapere quale account usa l'azienda (R27, R28);
  - **sbaglia in 5 casi** perché non conosce tutte le regole del chatbot: considera "non fedele" il rinvio al supporto tecnico (R11, R31) che invece è il comportamento previsto; vorrebbe una domanda su marca e modello del pad (R07, R08) dove la regola dice di dare istruzioni per ciascun caso; nella R30 suggerisce il comando `net use`, vietato dalla regola 2.
- **La sicurezza è il dato più solido:** 34 su 36, e le due bocciature (R09, R35) sono giuste. Nessuna risposta propone all'utente il registro di sistema, comandi o soluzioni che riducono la sicurezza.
- **I numeri del giudice non sono una misura affidabile:** come il giudice dei chunk non è tarato e, delle sue 19 bocciature, 5 sono sbagliate e 7 discutibili. È però un **buon secondo lettore**: indica dove guardare e ha trovato errori che erano sfuggiti. Nella fase 4 lo useremo così: il giudice segnala, una persona verifica.
- **La lezione per il giudice:** riceveva un riassunto delle regole e non le stesse regole del chatbot. Un giudice deve conoscere esattamente le regole che valuta: nella fase 4 del progetto gli passeremo le istruzioni del chatbot esattamente come sono, precisando che il rinvio al supporto è sempre consentito.
- **Correzioni per l'app, da verificare nella fase 4:** nell'input, "Codice di errore: non indicato dall'utente" invece di "nessuno"; nella regola 2, vietare anche di installare, disinstallare o riparare programmi e driver. Non le applichiamo ora per non eseguire di nuovo risposte e giudizi sulle stesse 36 richieste.

## 10. Riepilogo del ticket e riclassificazione (prompt 5)

Quando il problema non si risolve l'app apre il ticket. L'LLM scrive titolo e descrizione leggendo **tutta la conversazione**, che contiene più informazioni del primo messaggio: il dispositivo, i passi già provati, i codici di errore. Per lo stesso motivo **riclassifica** la categoria: è la classificazione che arriva al gruppo di supporto.

Simuliamo la conversazione più comune: la richiesta, l'eventuale domanda sul dispositivo, la risposta del chatbot e, se erano istruzioni, l'utente che risponde "ho provato, ma il problema c'è ancora". Negli altri esiti il ticket si apre subito.

In [27]:
class RiepilogoTicket(BaseModel):
    titolo: str = Field(description="Al massimo 10 parole, in italiano")
    descrizione: str = Field(description="In italiano: il problema, il dispositivo, cosa ha già provato l'utente, eventuali codici di errore")
    dispositivo: Literal[tuple(DISPOSITIVI)]
    categoria: Literal[tuple(CATEGORIE)]
    note_tecniche: str = Field(description="Cause e soluzioni da amministratore indicate dall'assistente, con le fonti; vuoto se non ce ne sono")


ISTRUZIONI_TICKET = f'''Sei l'help desk IT di primo livello. Il problema dell'utente non è stato risolto e devi aprire un ticket per il secondo livello.
Leggi la conversazione e le note dell'assistente, poi scrivi:
- un titolo breve e specifico;
- una descrizione chiara per il tecnico: il problema, il dispositivo, cosa ha già provato l'utente, eventuali codici di errore. Non inventare informazioni che non sono nella conversazione;
- il dispositivo;
- la categoria del ticket, scegliendo tra:
{elenco(DESCRIZIONE_CATEGORIE)}
- le note tecniche: riporta le cause e le soluzioni da amministratore indicate dall'assistente, con le fonti.'''


def conversazione(r):
    risposta = risposte[r["id"]]
    righe = [f"Utente: {r['richiesta_it']}"]
    if r["dispositivo_llm"] == "non chiaro":
        righe += ["Assistente: Di quale dispositivo si tratta?", f"Utente: {r['dispositivo_app']}"]
    righe.append(f"Assistente: {risposta.messaggio}")
    if risposta.esito == "istruzioni":
        righe.append("Utente: Ho provato tutti i passi, ma il problema c'è ancora.")
    risultati = trovati[(VARIANTE_SCELTA, r["id"])]
    elenco_fonti = "; ".join(f"[{n}] {risultati[n - 1]['titolo']}" for n in risposta.fonti
                             if 1 <= n <= len(risultati)) or "nessuna"
    return ("Conversazione:\n" + "\n".join(righe)
            + f"\n\nNote dell'assistente per il tecnico: {risposta.note_per_tecnico or 'nessuna'}"
            + f"\nFonti citate: {elenco_fonti}")


def apri_ticket(r):
    return chiedi("riepilogo del ticket", ISTRUZIONI_TICKET, conversazione(r), RiepilogoTicket)

In [28]:
%%time
tickets = dict(zip(richieste["id"], in_parallelo(apri_ticket, richieste.to_dict("records"))))
richieste["categoria_ticket"] = [tickets[i].categoria for i in richieste["id"]]
richieste["dispositivo_ticket"] = [tickets[i].dispositivo for i in richieste["id"]]

tre_classificazioni = pd.DataFrame({
    "categoria corretta": [
        (richieste["categoria_svm"] == richieste["categoria_attesa"]).sum(),
        (richieste["categoria_llm"] == richieste["categoria_attesa"]).sum(),
        (richieste["categoria_ticket"] == richieste["categoria_attesa"]).sum(),
    ]}, index=["SVM, primo messaggio", "LLM, primo messaggio", "LLM, intera conversazione (ticket)"])
display(tre_classificazioni)
print(f"Dispositivo del ticket uguale a quello atteso: {(richieste['dispositivo_ticket'] == richieste['dispositivo_atteso']).sum()} su {len(richieste)}")
display(richieste.loc[richieste["categoria_ticket"] != richieste["categoria_llm"],
                      ["id", "richiesta_it", "categoria_attesa", "categoria_llm", "categoria_ticket"]])

,categoria corretta
"SVM, primo messaggio",19
"LLM, primo messaggio",32
"LLM, intera conversazione (ticket)",33


Dispositivo del ticket uguale a quello atteso: 36 su 36


,id,richiesta_it,categoria_attesa,categoria_llm,categoria_ticket
19,R20,Non riesco a fare la scansione su cartella dalla multifunzione,Hardware,Storage,Hardware


CPU times: total: 15.6 ms
Wall time: 20 ms


In [29]:
for id_richiesta in ["R18", "R30"]:
    ticket = tickets[id_richiesta]
    print(f"{id_richiesta} · {ticket.titolo}\nDispositivo: {ticket.dispositivo} | categoria: {ticket.categoria}\n")
    print(ticket.descrizione)
    if ticket.note_tecniche:
        print(f"\nNote tecniche: {ticket.note_tecniche}")
    print("\n" + "-" * 100)

R18 · Lexmark richiede la sostituzione del toner appena cambiato
Dispositivo: stampante di rete | categoria: Hardware

La stampante Lexmark segnala di cambiare il toner, anche se l'utente lo ha appena sostituito. Non è stato specificato il messaggio esatto visualizzato sul display. Non risultano altre prove effettuate o codici di errore.

----------------------------------------------------------------------------------------------------
R30 · Impossibile aprire la cartella condivisa dell’ufficio
Dispositivo: computer | categoria: Storage

L’utente non riesce ad aprire la cartella condivisa dell’ufficio. Non sono stati forniti codici di errore né ulteriori dettagli sui tentativi già effettuati. L’assistente ha consigliato di contattare il supporto IT per verificare l’accesso.

Note tecniche: La causa non è identificabile senza codice di errore o altri dettagli. Le soluzioni citate richiedono intervento amministrativo sui servizi di Windows ([3], [4]); per l’errore 0x80004005 sono menzi

### Osservazioni (parte 10)

| Classificazione | Categorie corrette |
|---|---|
| SVM, primo messaggio | 19 su 36 |
| LLM, primo messaggio | 32 su 36 |
| LLM, intera conversazione (ticket) | 33 su 36 |

- **La riclassificazione è stabile:** leggendo tutta la conversazione cambia una sola categoria su 36 (R20, scansione su cartella: da Storage a Hardware) e nessuna peggiora. Il +1 però non dimostra un miglioramento: la R20 era una delle etichette discutibili della sezione 4 e il cambio dipende dalla domanda del chatbot sulla marca della multifunzione. Con richieste di una sola frase è prevedibile; la riclassificazione conterà di più nelle conversazioni lunghe della fase 4.
- **Il dispositivo del ticket è giusto in 36 casi su 36** compresi i due "non chiaro" (R01, R24), risolti dalla risposta dell'utente nella conversazione.
- **I ticket sono chiari e non inventano:** nella R18 il ticket dice esplicitamente che il messaggio esatto sul display non è stato indicato; nella R30 le note tecniche riportano le cause possibili, le soluzioni da amministratore e l'avvertenza di non applicare accesso guest e SMB 1.0.
- **Un difetto da correggere nell'app:** le note per il tecnico citano gli estratti con il loro numero ([1]-[5]), ma al ticket passiamo solo le fonti citate nel messaggio all'utente. Nella R30 nessuna e il ticket chiude con "Fonti bibliografiche non fornite". Nell'app il ticket allegherà tutti gli estratti trovati, con titolo, pagina e link.
- **Un limite della simulazione:** quando il chatbot fa una domanda (per esempio nella R18), il ticket viene aperto senza la risposta dell'utente. Nell'app il ticket si aprirà solo dopo la risposta.

## 11. Costi e tempi

Tutte le chiamate sono nella cache, con i token e i secondi. Consideriamo solo quelle di questa esecuzione: se nel frattempo abbiamo cambiato un prompt, le risposte del prompt vecchio restano nel file ma non contano. Calcoliamo il costo per compito e, soprattutto, quanto costa e quanto dura una conversazione nell'app: analisi, eventuale HyDE, risposta ed eventuale ticket. I giudizi non ci sono perché servono solo alla valutazione.

I tempi sono quelli di una singola chiamata (mediana), misurati mentre altre chiamate erano in corso: nell'app, con un utente alla volta, potrebbero essere un po' più bassi.

In [30]:
tutte = pd.DataFrame(cache_llm.values())
tutte["costo"] = tutte.apply(costo, axis=1)
chiamate = tutte[tutte["chiave"].isin(chiavi_usate)]   # solo le chiamate di questa esecuzione, non quelle di prompt vecchi
per_compito = chiamate.groupby(["compito", "modello"]).agg(
    chiamate=("chiave", "count"),
    token_ingresso_medi=("token_ingresso", "mean"),
    token_uscita_medi=("token_uscita", "mean"),
    di_cui_ragionamento=("token_ragionamento", "mean"),
    secondi_mediani=("secondi", "median"),
    costo_totale=("costo", "sum"),
).round({"token_ingresso_medi": 0, "token_uscita_medi": 0, "di_cui_ragionamento": 0, "secondi_mediani": 1, "costo_totale": 4})
display(per_compito)
print(f"Costo delle chiamate di questo notebook: circa {chiamate['costo'].sum():.2f} dollari "
      f"(speso in tutto, comprese le prove con prompt precedenti: {tutte['costo'].sum():.2f})")

compiti_app = ["analisi", "risposta", "riepilogo del ticket"] + (["HyDE"] if USA_HYDE else [])
medie = per_compito.reset_index().set_index("compito").loc[compiti_app]
costo_conversazione = (medie["costo_totale"] / medie["chiamate"]).sum()
attesa_prima_risposta = medie.drop(index="riepilogo del ticket")["secondi_mediani"].sum()
print(f"Una conversazione nell'app ({', '.join(compiti_app)}): circa {costo_conversazione * 1000:.2f} dollari ogni 1.000 conversazioni "
      f"-> {costo_conversazione:.5f} dollari l'una")
print(f"Attesa prima della risposta (chiamate in sequenza): circa {attesa_prima_risposta:.1f} secondi")

,,chiamate,token_ingresso_medi,token_uscita_medi,di_cui_ragionamento,secondi_mediani,costo_totale
compito,modello,,,,,,
HyDE,gpt-6-luna,36,181.0,90.0,0.0,1.6,0.0023
analisi,gpt-6-luna,36,871.0,48.0,0.0,1.3,0.0040
giudizio dei chunk,gpt-6.1-terra,36,1239.0,98.0,29.0,2.8,0.0622
giudizio della risposta,gpt-6.1-terra,36,1622.0,173.0,63.0,3.3,0.0896
riepilogo del ticket,gpt-6-luna,36,805.0,165.0,0.0,2.2,0.0059
risposta,gpt-6-luna,36,1684.0,238.0,70.0,2.7,0.0104


Costo delle chiamate di questo notebook: circa 0.17 dollari (speso in tutto, comprese le prove con prompt precedenti: 0.34)
Una conversazione nell'app (analisi, risposta, riepilogo del ticket, HyDE): circa 0.63 dollari ogni 1.000 conversazioni -> 0.00063 dollari l'una
Attesa prima della risposta (chiamate in sequenza): circa 5.6 secondi


### Osservazioni (parte 11)

- **Costo del notebook:** circa 0,17 dollari per le chiamate di questa esecuzione, quasi tutti (0,15) per il giudice `gpt-6.1-terra`; i compiti del chatbot con `gpt-6-luna` costano in tutto circa 2 centesimi. Comprese le prove con i prompt precedenti (versione 1, esecuzione con il "nan", "brani"), la spesa totale è di 0,34 dollari.
- **Costo di una conversazione nell'app:** circa 0,0006 dollari (analisi, HyDE, risposta e riepilogo del ticket), cioè circa 1.600 conversazioni per un dollaro. La voce più cara è la risposta (circa 0,0003 dollari), che legge i 5 estratti (circa 1.700 token in ingresso); gli embeddings per la ricerca sono trascurabili.
- **Attesa prima della risposta:** circa 5,6 secondi (analisi 1,3 s, HyDE 1,6 s, risposta 2,7 s, in sequenza), più pochi decimi di secondo per la ricerca. Il riepilogo del ticket (2,2 s) arriva dopo, solo quando serve.
- **Ragionamento:** la risposta usa in media 70 token di ragionamento (sforzo `low`); analisi, HyDE e ticket nessuno (sforzo `none`).
- **HyDE costa poco ma allunga l'attesa di circa un quarto** (1,6 secondi su 5,6). Nella fase 4 proveremo a far scrivere il passaggio ipotetico nella stessa chiamata dell'analisi, verificando che la qualità della ricerca non cambi.
- I tempi sono la mediana di chiamate fatte in parallelo: nell'app, con un utente alla volta, potrebbero essere leggermente diversi.

## 12. Scelte per l'app e salvataggio

Decidiamo con regole scritte come nei notebook precedenti:

- **categoria del ticket:** l'LLM sostituisce l'SVM solo se indovina almeno 3 richieste in più: con 36 richieste, una differenza di 1-2 è rumore (notebook 06, sezione 6). L'SVM resta comunque nel progetto come modello di machine learning addestrato e valutato nei notebook 02-04;
- **HyDE:** la variante scelta nella sezione 7;
- **modelli e sforzi:** quelli usati qui.

Salviamo tutto in `configurazione_llm.json`, prompt compresi: l'app userà esattamente i prompt valutati in questo notebook. Quando scriveremo il codice dell'app in `src/`, i prompt diventeranno un file Python sotto controllo di versione.

Prompt e scelte sono stati valutati sulle stesse 36 richieste, nella fase 4 proveremo l'app con richieste **nuove**.

In [31]:
corrette_llm = int(giusta_llm.sum())
corrette_svm = int(giusta_svm.sum())
CLASSIFICATORE = "llm" if corrette_llm - corrette_svm >= 3 else "svm"
print(f"Categoria: LLM {corrette_llm} contro SVM {corrette_svm} -> nell'app usiamo: {CLASSIFICATORE.upper()}")

configurazione_llm = {
    "modello_llm": MODELLO_LLM,
    "modello_giudice": MODELLO_GIUDICE,
    "sforzo": {"analisi": "none", "hyde": "none", "risposta": "low", "ticket": "none"},
    "classificatore_categoria": CLASSIFICATORE,
    "usa_hyde": USA_HYDE,
    "variante_ricerca": VARIANTE_SCELTA,
    "istruzioni": {
        "analisi": ISTRUZIONI_ANALISI,
        "hyde": ISTRUZIONI_HYDE,
        "risposta": ISTRUZIONI_RISPOSTA,
        "ticket": ISTRUZIONI_TICKET,
    },
    "descrizione_dispositivi": DESCRIZIONE_DISPOSITIVI,
    "descrizione_categorie": DESCRIZIONE_CATEGORIE,
    "risultati": {
        "categoria_corretta": {"svm": corrette_svm, "llm": corrette_llm,
                               "ticket": int((richieste["categoria_ticket"] == richieste["categoria_attesa"]).sum())},
        "dispositivo": {esito: int(n) for esito, n in richieste["esito_dispositivo"].value_counts().items()},
        "giudice_chunk": {"accordo": round(float(accordo), 3), "kappa": round(float(kappa), 3)},
        "risposte": {"fedeli": int(richieste["fedele"].sum()), "sicure": int(richieste["sicura"].sum()),
                     "esito_appropriato": int(richieste["esito_appropriato"].sum()),
                     "utilita_media": round(float(richieste["utilita"].mean()), 2)},
        "costo_conversazione_dollari": round(float(costo_conversazione), 6),
    },
}
with open(CONFIG_LLM_PATH, "w", encoding="utf-8") as f:
    json.dump(configurazione_llm, f, indent=2, ensure_ascii=False)
print("Salvata:", CONFIG_LLM_PATH)

Categoria: LLM 32 contro SVM 19 -> nell'app usiamo: LLM
Salvata: ..\models\configurazione_llm.json


## 13. Riepilogo

| Aspetto | Valore / osservazione |
|---|---|
| Modelli (chatbot / giudice) e costo del notebook | `gpt-6-luna` / `gpt-6.1-terra`; 0,17 dollari per questa esecuzione (quasi tutti per il giudice), 0,34 comprese le prove con i prompt precedenti |
| Traduzione dell'LLM contro quella manuale (ricerca) | equivalenti: risultati identici con il filtro; senza filtro le stesse 34 richieste trovate, MRR 0,931 contro 0,944 |
| Dispositivo: giusto / non chiaro / sbagliato | 34 / 2 / 0: quando la richiesta è ambigua (R01, R24) il chatbot chiede invece di indovinare |
| Categoria: SVM / LLM / LLM sull'intera conversazione | 19 / 32 / 33. Nessuna richiesta è giusta solo per l'SVM; dei 4 disaccordi, 3 sono etichette discutibili su cui LLM e SVM concordano |
| Ricerca come nell'app (configurazione C) | 36 su 36, sempre al primo posto, precisione 94,4%: come la configurazione ideale del notebook 06, senza traduzioni manuali e senza conoscere in anticipo il dispositivo |
| Giudice dei chunk: accordo e kappa con i nostri giudizi | 36%, kappa 0,19: sotto la soglia di 0,6, quindi non usato; HyDE valutato leggendo i chunk a mano |
| HyDE: effetto e variante scelta | T+H: 12 richieste migliori, 4 peggiori, nessun peggioramento netto (R03, R21, R26 migliorano nettamente). HyDE da solo (H) confonde le marche (R16, R18): la traduzione conserva la marca citata dall'utente |
| Risposte: esiti | versione 1: 15 istruzioni, 15 domande, 5 inoltri, 1 non trovata; versione 2: 21 / 7 / 5 / 3. Con estratti utili (i nostri "sì") istruzioni in 16 casi su 17 |
| Risposte: fedeli / sicure / esito appropriato / utilità | 22 / 34 / 26 su 36, utilità 3,7 su 5 secondo il giudice. Delle 19 bocciature, 7 giuste, 7 in parte, 5 sbagliate: è un buon secondo lettore, non una misura affidabile. La sicurezza è il dato più solido |
| Costo e attesa per conversazione | circa 0,0006 dollari (circa 1.600 conversazioni per dollaro); circa 5,6 secondi prima della risposta, di cui 1,6 per HyDE |
| Scelte per l'app | `gpt-6-luna` per tutti i compiti; analisi in una sola chiamata; ricerca T+H con filtro per dispositivo; categoria dall'LLM; prompt della risposta versione 2; ticket con riclassificazione sull'intera conversazione |

### Prossimo passo

Il codice di questi notebook diventa un'applicazione: in `src/` scriveremo i moduli per la ricerca, le chiamate all'LLM e il flusso della conversazione, poi l'interfaccia con Streamlit, con i pulsanti per il dispositivo e per la marca della stampante di rete. Nell'app porteremo le correzioni emerse qui:

- nell'input della risposta, "Codice di errore: non indicato dall'utente" invece di "nessuno";
- nella regola 2, vietare anche di installare, disinstallare o riparare programmi e driver;
- nel ticket, allegare tutti gli estratti trovati, così le note per il tecnico si possono verificare;
- al giudice, le stesse regole del chatbot, precisando che il rinvio al supporto è sempre consentito.

Nella fase 4 proveremo l'app con richieste nuove: è lì che misureremo davvero i risultati,